# Test final de la fusión 3D (modelos B) — cuaderno para correr en el equipo con GPU

**Qué hace:** carga las definiciones del cuaderno de entrenamiento (sin entrenar nada), pasa la fusión a los pesos de la **etapa B** (`best_B_decoder.pth`) y evalúa el split **test** una sola vez.

**Requisitos en este equipo:** el repositorio con `data_mha/` y `splits/dataset.csv`, y las carpetas `resultados/modelos/entrenamiento_multitarea_{axial,coronal,sagital}/best_B_decoder.pth`.

**Cómo correrlo:** abrir en el mismo repositorio que el cuaderno de entrenamiento (para que encuentre `requirements.txt` y `src/`), reiniciar el kernel y ejecutar todas las celdas en orden (*Run All*). **No hay ninguna celda que entrene.** Las celdas 1–8 son las definiciones del cuaderno original; la última es el test y solo debe correrse una vez.

**Al terminar, enviar:** este cuaderno con las salidas visibles y el archivo `resultados/resultados_test_fusion.zip`.

## 1. Definiciones del cuaderno de entrenamiento (sin entrenar)

In [1]:
import copy, gc, hashlib, json, math, random, shutil, subprocess, sys, time, ast, traceback
from collections import defaultdict
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
import SimpleITK as sitk
from scipy.ndimage import maximum_filter
from skimage.filters import threshold_otsu            # los usa enhance_bone_contrast (Otsu + CLAHE del preprocesamiento)
from skimage.exposure import equalize_adapthist

# ------------------------------------------------------------------ rutas
# Raiz del repositorio: funciona si el kernel arranca en notebooks/ o en la raiz.
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "requirements.txt").is_file())
BASE_DIR = ROOT                 # nombre global que usan las definiciones del equipo
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
try:
    from src.rutas import CACHE_DIR, DATASET_CSV, DATOS_DIR, FIGURAS_DIR, METRICAS_DIR, MODELOS_DIR, NOTEBOOKS_DIR
except ModuleNotFoundError:     # repositorio sin src/rutas.py: mismas carpetas
    NOTEBOOKS_DIR, DATASET_CSV = ROOT / "notebooks", ROOT / "splits" / "dataset.csv"
    DATOS_DIR, METRICAS_DIR, FIGURAS_DIR = ROOT / "resultados" / "datos", ROOT / "resultados" / "metricas", ROOT / "resultados" / "figuras"
    MODELOS_DIR, CACHE_DIR = ROOT / "resultados" / "modelos", ROOT / "resultados" / "cache"

# ------------------------------------------------------------------ qué correr
MODO = "completo"                   # "auto": prueba corta de los planos y luego la corrida completa | "completo": solo la completa | "prueba": solo la prueba corta
PLANOS_A_ENTRENAR = ["coronal", "sagital", "axial"]     # en este orden; los dos nuevos primero (el axial ya tiene un modelo en uso)
FORCE_RETRAIN = False           # False: retoma lo ya hecho (modelos exportados y etapas completas) | True: rehace todo
BORRAR_CACHE_AL_TERMINAR = False    # True: borra las cachés de un plano al exportar su modelo (ahorra disco; reconstruirlas tarda)

# ------------------------------------------------------------------ datos y modelo
SEED = 42
IMAGE_SIZE = 256
GRID_SIZE = 16
USE_25D = True                  # True: canales [i-1, i, i+1] (como el modelo axial 2.5D del equipo) | False: solo el corte i
CENTER_MODE = "interior"        # centro objetivo del decoder: "interior" (siempre sobre el hueso) | "mean" (versión anterior)
MIN_PIXELS = 20                 # píxeles mínimos de una región para contar un corte como positivo (mismo criterio que Targets_deteccion)
CACHE_IMG_DTYPE = "float16"     # "float16" (igual que el equipo) | "uint8" (mitad de disco y de RAM; cuantiza la imagen a 256 niveles)
BACKBONE_NAME = "plus"          # "base" | "plus"
USE_CBAM = True
USE_SKIPS = True                # decoder con skip connections del backbone; False = baseline tipo FCN
USE_CBAM_GATE = True            # True: compuerta residual y = x + gamma*(CBAM(x)-x), gamma inicia en 0 (diseño del equipo)

BATCH_SIZE = 16
VAL_BATCH_SIZE = 16
NUM_WORKERS = 0                 # debe ser 0: set_epoch() (línea media del canal de posición) no llega a procesos de DataLoader persistentes
WEIGHT_DECAY = 1e-4
GRAD_CLIP = 10.0                # norma máxima del gradiente
SEG_WEIGHTS = (1.0, 1.0, 1.0)   # semantic, center, offset

DET_LAMBDAS_SOURCE = "loaded"   # "loaded" (los LAMBDAS de deteccion_grid.py) | "afinados" (copia local de abajo)
LAMBDAS_AFINADOS = {"obj": 0.46, "box": 2.90, "cls": 2.03, "noobj": 1.0, "l1": 1.0, "iou": 1.0}

JITTER_PX = 8.0                 # desplazamiento máximo de cada coordenada de caja (limitado al 20% del lado)
PRED_BOX_SCORE_MIN = 0.05       # score mínimo para aceptar una caja predicha como prior
INSTANCE_EVAL_N = 128           # cortes de val para la métrica de instancias (decodificación lenta)
EVAL_SCORE_MIN = 0.01           # score mínimo de candidatas en el AP
LOG_GRAD_EVERY = 50             # cada cuántas iteraciones medir normas de gradiente por módulo

# ------------------------------------------------------------------ canal de posición (solo sagital)
POS_SIGMA_MM = 15.0             # ruido (σ) sobre la línea media que ve la red en entrenamiento (el centro del FOV se aleja del sacro 14.4 mm de media, en los 100 casos)
POS_P_FOV = 0.3                 # probabilidad de usar el centro del campo de visión como línea media en entrenamiento
POS_VAL_MODE = "fov"            # línea media que ve val durante el entrenamiento: "fov" (realista) | "true" (cota superior)

# ------------------------------------------------------------------ cortes negativos (sin un solo vóxel de pelvis)
USE_NEGATIVOS = True
NEG_POR_POSITIVO = 1 / 3        # negativos de train por cada corte positivo de train (en axial, los negativos naturales son ~0.32 por positivo)
NEG_CERCANOS = 0.6              # fracción de los negativos elegidos que sale de la mitad más cercana a la pelvis de cada caso
UMBRAL_REFERENCIA = 0.08        # umbral_corte vigente del proyecto (solo para la columna *_fijo de las métricas de cortes vacíos)
RECALL_MIN = 0.99               # recall de pelvis que debe conservar el umbral recalibrado
SEL_PESO_VACIOS = 1 / 3         # peso de (1 - cortes vacíos que pasan) en el puntaje de selección de checkpoint (etapas A y C)

STAGES_BASE = [
    {"name": "A_deteccion", "epochs": 10, "lr": 1e-3,
     "train": ["backbone", "cbam", "det_head"],
     "w_det": 1.0, "w_seg": 0.0, "prior": "gt", "select": "det"},
    {"name": "B_decoder", "epochs": 10, "lr": 1e-3,
     "train": ["seg_decoder"],
     "w_det": 0.0, "w_seg": 1.0, "prior": "gt_jitter", "select": "seg_gt"},
    {"name": "C_conjunto", "epochs": 10, "lr": 3e-4,
     "train": ["backbone", "cbam", "det_head", "seg_decoder"],
     "w_det": 1.0, "w_seg": 1.0, "prior": "mixed", "p_pred_end": 0.5, "select": "both"},
]
AJUSTES_PRUEBA = {"MAX_TRAIN_SLICES": 64, "MAX_VAL_SLICES": 32, "MAX_NEG_SLICES": 32, "INSTANCE_EVAL_N": 8,
                  "BATCH_SIZE": 8, "VAL_BATCH_SIZE": 8, "epochs": 1}
_BASE = {"BATCH_SIZE": BATCH_SIZE, "VAL_BATCH_SIZE": VAL_BATCH_SIZE, "INSTANCE_EVAL_N": INSTANCE_EVAL_N}

assert MODO in ("auto", "completo", "prueba"), MODO
assert NUM_WORKERS == 0, "NUM_WORKERS debe ser 0 (ver la nota de la configuración)"
assert CACHE_IMG_DTYPE in ("float16", "uint8")

# ------------------------------------------------------------------ dispositivo y utilidades
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
RAIZ_LOG = METRICAS_DIR / "entrenamiento_multitarea_planos"
RAIZ_LOG.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = RAIZ_LOG / "log_corrida.txt"


def log(msg=""):
    """Imprime y añade al registro de la corrida (resultados/metricas/entrenamiento_multitarea_planos/log_corrida.txt)."""
    line = f"[{datetime.now():%Y-%m-%d %H:%M:%S}] {msg}"
    print(line, flush=True)
    with open(LOG_PATH, "a", encoding="utf-8") as fh:
        fh.write(line + "\n")


def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False


def autocast():
    return torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP)


seed_everything(SEED)
print(f"PyTorch {torch.__version__} | dispositivo: {DEVICE}"
      + (f" ({torch.cuda.get_device_name(0)})" if DEVICE.type == "cuda" else "")
      + f" | AMP: {USE_AMP} | MODO: {MODO} | planos: {PLANOS_A_ENTRENAR}")
print("Registro de la corrida:", LOG_PATH)

PyTorch 2.11.0+cu128 | dispositivo: cuda (NVIDIA GeForce RTX 5060) | AMP: True | MODO: completo | planos: ['coronal', 'sagital', 'axial']
Registro de la corrida: D:\proyecto_analitica_corte2\resultados\metricas\entrenamiento_multitarea_planos\log_corrida.txt


In [2]:
def _target_names(node):
    targets = node.targets if isinstance(node, ast.Assign) else [node.target] if isinstance(node, ast.AnnAssign) else []
    names = set()
    for target in targets:
        if isinstance(target, ast.Name):
            names.add(target.id)
        elif isinstance(target, (ast.Tuple, ast.List)):
            names.update(item.id for item in target.elts if isinstance(item, ast.Name))
    return names


def _load_existing_definitions(notebook_path, definition_names, value_names=()):
    notebook_path = Path(notebook_path)
    if not notebook_path.exists():
        raise FileNotFoundError(f"No encuentro el cuaderno del equipo: {notebook_path}")
    notebook = json.loads(notebook_path.read_text(encoding="utf-8"))
    wanted_defs, wanted_values = set(definition_names), set(value_names)
    snippets, found_defs, found_values = [], set(), set()
    for cell in notebook.get("cells", []):
        if cell.get("cell_type") != "code":
            continue
        source = "".join(cell.get("source", []))
        try:
            tree = ast.parse(source)
        except SyntaxError:
            continue
        for node in tree.body:
            name = getattr(node, "name", None)
            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)) and name in wanted_defs and name not in found_defs:
                snippets.append(ast.get_source_segment(source, node)); found_defs.add(name)
            elif isinstance(node, (ast.Assign, ast.AnnAssign)):
                selected = _target_names(node) & wanted_values
                if selected and not selected.issubset(found_values):
                    snippets.append(ast.get_source_segment(source, node)); found_values.update(selected)
    missing_defs, missing_values = wanted_defs - found_defs, wanted_values - found_values
    if missing_defs or missing_values:
        raise RuntimeError(f"Faltan definiciones en {notebook_path.name}: "
                           f"funciones/clases={sorted(missing_defs)}, constantes={sorted(missing_values)}")
    exec("\n\n".join(snippets), globals())
    return len(snippets)


try:                                                         # preferido: conserva decoradores (@torch.no_grad) y ejecuta los imports de cada celda
    from src.definiciones_notebooks import cargar_definiciones as _cargar_src

    def cargar(nb, definiciones, valores=()):
        return _cargar_src(nb, set(definiciones), valores=set(valores), destino=globals())
except ModuleNotFoundError:                                  # sin src/definiciones_notebooks.py: el cargador de arriba
    def cargar(nb, definiciones, valores=()):
        return _load_existing_definitions(nb, set(definiciones), set(valores))


n_def = 0
n_def += cargar(NOTEBOOKS_DIR / "Dataset_splits.ipynb", {"resolve_repo_path"})
n_def += cargar(NOTEBOOKS_DIR / "Ventaneo_hu.ipynb", {"apply_hu_window", "enhance_bone_contrast"})
n_def += cargar(NOTEBOOKS_DIR / "Targets_deteccion.ipynb",
                {"instance_to_semantic", "boxes_from_semantic", "letterbox_pair", "build_detection_target", "PengwinDetectionDataset"},
                {"CLASS_ID_TO_NAME", "TARGET_COLORS"})
n_def += cargar(NOTEBOOKS_DIR / "Backbone.ipynb", {"FundidoraPC", "FundidoraPCPlus", "build_backbone"})
n_def += cargar(NOTEBOOKS_DIR / "Segmentacion_instancias.ipynb",
                {"build_instance_segmentation_targets", "build_detection_region_priors", "SegmentationConvBlock",
                 "backbone_feature_pyramid", "SkipFusionBlock", "CenterOffsetSkipDecoder", "center_offset_segmentation_loss",
                 "soft_dice_semantic_loss", "decode_center_offset_instances"})
n_def += cargar(NOTEBOOKS_DIR / "cbam.ipynb", {"ChannelAttention", "SpatialAttention", "CBAM", "CompuertaResidualCBAM"})

from src.deteccion_grid import (
    box_iou, pairwise_iou, encode_boxes, decode_boxes, build_targets, GridDetectionHead,
    split_head_output, grid_detection_loss, decode_predictions, nms_por_clase, postprocess,
    GRID_S, IMG_SIZE, STRIDE, CH_OBJ, CH_BOX, CH_CLS, LAMBDAS, CLASS_NAMES,
)
from src.pipeline_pengwin import CONFIG_PIPELINE
from src.entrada_25d import indices_vecinos
from src import cortes_vacios as cv
from src import orientacion, planos
from src.orientacion import load_canonical
from src.planos import (PLANES, make_sample, positive_slices, negative_slices, sample_negatives,
                        midline_mm, fov_center_mm, slice_position_mm, position_value, neighbor_indices)
print(f"Definiciones de los cuadernos del equipo: {n_def} | src: deteccion_grid, pipeline_pengwin, entrada_25d, cortes_vacios, orientacion, planos")
assert GRID_S == GRID_SIZE and IMG_SIZE == IMAGE_SIZE, "la cabeza grid espera otra resolución"

# --- pesos de la pérdida de detección ---
DET_LAMBDAS = dict(LAMBDAS) if DET_LAMBDAS_SOURCE == "loaded" else dict(LAMBDAS_AFINADOS)
print(f"Pesos de la pérdida de detección ({DET_LAMBDAS_SOURCE}):", DET_LAMBDAS)
if dict(LAMBDAS) != LAMBDAS_AFINADOS:
    print("AVISO: LAMBDAS de deteccion_grid.py y los afinados no coinciden.")

# --- tablas que usa PengwinDetectionDataset (variables globales df y slice_manifest) ---
df = pd.read_csv(DATASET_CSV, dtype={"case_id": str})
_MANIFEST_AXIAL_EQUIPO = DATOS_DIR / "targets" / "manifest_cortes_positivos.csv"      # solo para la comprobación de equivalencia
slice_manifest = pd.read_csv(_MANIFEST_AXIAL_EQUIPO, dtype={"case_id": str}) if _MANIFEST_AXIAL_EQUIPO.exists() else None
print("Casos por split:", df["split"].value_counts().to_dict())

Definiciones de los cuadernos del equipo: 26 | src: deteccion_grid, pipeline_pengwin, entrada_25d, cortes_vacios, orientacion, planos
Pesos de la pérdida de detección (loaded): {'obj': 0.46, 'box': 2.9, 'cls': 2.03, 'noobj': 1.0, 'l1': 1.0, 'iou': 1.0}
Casos por split: {'train': 75, 'test': 20, 'val': 5}


In [3]:
class MultiTaskNet(nn.Module):
    def __init__(self, backbone_name="base", use_cbam=True, use_gate=True, use_skips=True, in_channels=1, image_channels=None):
        super().__init__()
        self.image_channels = int(in_channels if image_channels is None else image_channels)   # canales de imagen (sin posición)
        self.backbone = build_backbone(backbone_name, in_channels=in_channels)
        if not use_cbam:
            self.cbam = None
        else:
            # Compuerta residual (gamma inicia en 0): al empezar, el tronco es idéntico al backbone sin atención
            self.cbam = CompuertaResidualCBAM(CBAM(256)) if use_gate else CBAM(256)
        self.det_head = GridDetectionHead()
        self.seg_decoder = CenterOffsetSkipDecoder(in_channels=256, image_channels=self.image_channels, use_skips=use_skips)

    def trunk(self, x):
        # Pirámide [f1 (32,128²), f2 (64,64²), f3 (128,32²), f4 (256,16²)]; f4 == backbone._encode(x) (incluye el contexto de 'plus').
        feats = backbone_feature_pyramid(self.backbone, x)
        if self.cbam is not None:
            feats[-1], _, _ = self.cbam(feats[-1])  # CBAM devuelve (y, atención de canal, atención espacial); solo se refina f4
        return feats

    def segment(self, x, feats, region_priors):
        # El decoder recibe solo los canales de imagen; el canal de posición (si existe) va al final y no se le pasa.
        return self.seg_decoder(x[:, :self.image_channels], feats, region_priors)

    def forward(self, x, region_priors=None):
        feats = self.trunk(x)
        det_logits = self.det_head(feats[-1])
        seg_out = self.segment(x, feats, region_priors) if region_priors is not None else None
        return det_logits, seg_out


def count_params(module):
    return sum(p.numel() for p in module.parameters())


def construir_modelo(verbose=True):
    model = MultiTaskNet(BACKBONE_NAME, USE_CBAM, USE_CBAM_GATE, USE_SKIPS, IN_CHANNELS, N_IMG).to(DEVICE)
    if verbose:
        for name, child in model.named_children():
            print(f"{name:12s} {count_params(child):>10,} parámetros")
        print(f"{'TOTAL':12s} {count_params(model):>10,}")
    # El conteo de referencia del equipo (693,290 + γ) es con 1 canal de entrada y el backbone base.
    if BACKBONE_NAME == "base" and USE_CBAM and IN_CHANNELS == 1:
        _det = sum(count_params(getattr(model, n)) for n in ("backbone", "cbam", "det_head") if getattr(model, n) is not None)
        assert _det == 693_290 + (1 if USE_CBAM_GATE else 0), "el conteo de parámetros no coincide con las piezas del equipo"
    # El tronco con pirámide debe producir EXACTAMENTE el mapa f4 que usaba la cabeza de detección (backbone -> CBAM).
    with torch.no_grad():
        probe = torch.rand(2, IN_CHANNELS, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE)
        model.eval()
        _, fmap_old = model.backbone(probe)
        if model.cbam is not None:
            fmap_old, _, _ = model.cbam(fmap_old)
        assert torch.allclose(model.trunk(probe)[-1], fmap_old, atol=1e-5), "el tronco ya no coincide con backbone -> CBAM"
        model.train()
    return model

In [4]:
TARGETS_DIR = DATOS_DIR / "targets"
TARGETS_DIR.mkdir(parents=True, exist_ok=True)
SPLIT_DE_CASO = dict(zip(df["case_id"], df["split"]))


def ruta_manifiesto(plane, kind):
    return TARGETS_DIR / (f"manifest_cortes_{plane}.csv" if kind == "pos" else f"manifest_negativos_{plane}.csv")


def build_plane_manifests(planes=PLANES):
    """Genera los manifiestos que falten (positivos y negativos de los planos pedidos). Cada etiqueta se lee UNA vez, en orientación canónica."""
    faltan = [(p, k) for p in planes for k in ("pos", "neg") if not ruta_manifiesto(p, k).exists()]
    if not faltan:
        return
    filas = {key: [] for key in faltan}
    t0 = time.perf_counter()
    for n, r in enumerate(df.itertuples(index=False), 1):
        L, sp, _ = load_canonical(r.label_path, is_label=True, resolver=resolve_repo_path, dtype="native")
        mid_true, fov = midline_mm(L, sp), fov_center_mm(L.shape, sp)
        for p, kind in faltan:
            recs = positive_slices(L, p, sp, MIN_PIXELS, CLASS_ID_TO_NAME) if kind == "pos" else negative_slices(L, p)
            for rec in recs:
                filas[(p, kind)].append({"split": SPLIT_DE_CASO[r.case_id], "case_id": r.case_id, **rec,
                                         "x_mm": slice_position_mm(p, rec["slice_index"], sp),
                                         "mid_true_mm": mid_true, "fov_mm": fov, "sx": sp[0], "sy": sp[1], "sz": sp[2]})
        del L
        gc.collect()
        if n % 20 == 0 or n == len(df):
            log(f"manifiestos {sorted(set(faltan))}: {n}/{len(df)} casos | {time.perf_counter() - t0:.0f} s")
    for (p, kind), rows in filas.items():
        pd.DataFrame(rows).to_csv(ruta_manifiesto(p, kind), index=False)


def leer_manifiestos(plane):
    pos = pd.read_csv(ruta_manifiesto(plane, "pos"), dtype={"case_id": str})
    neg = pd.read_csv(ruta_manifiesto(plane, "neg"), dtype={"case_id": str})
    return pos, neg


def verificar_manifiestos(plane):
    """Ningún corte es positivo y negativo a la vez; el split de cada fila coincide con dataset.csv; sin fuga de pacientes."""
    pos, neg = leer_manifiestos(plane)
    cruce = pos.merge(neg, on=["case_id", "slice_index"], how="inner")
    assert cruce.empty, f"{plane}: {len(cruce)} cortes están en los dos manifiestos"
    for nombre, m in (("positivos", pos), ("negativos", neg)):
        assert (m["split"] == m["case_id"].map(SPLIT_DE_CASO)).all(), f"{plane}: el split de algún corte de {nombre} no coincide con dataset.csv"
    casos = {s: set(pos.loc[pos["split"] == s, "case_id"]) | set(neg.loc[neg["split"] == s, "case_id"]) for s in ("train", "val", "test")}
    assert not (casos["train"] & casos["val"]) and not (casos["train"] & casos["test"]) and not (casos["val"] & casos["test"]), "fuga de pacientes entre splits"
    resumen = pd.DataFrame({"positivos": pos.groupby("split").size(), "negativos": neg.groupby("split").size()}).fillna(0).astype(int)
    resumen["negativos_por_positivo"] = (resumen["negativos"] / resumen["positivos"]).round(2)
    log(f"manifiestos {plane}: " + " | ".join(f"{s}: {int(r.positivos)} pos, {int(r.negativos)} neg" for s, r in resumen.iterrows()))
    return resumen

In [5]:
def _sha1(path, n=12):
    h = hashlib.sha1()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()[:n]


def _etiqueta_cache(plane, split, kind, max_slices):
    return f"{plane}_{kind}_{split}" + (f"_first{max_slices}" if max_slices else "") + ("_25d" if N_IMG == 3 else "")


def _bytes_por_corte():
    return N_IMG * IMAGE_SIZE ** 2 * (2 if CACHE_IMG_DTYPE == "float16" else 1) + IMAGE_SIZE ** 2


def pad_boxes(boxes_list, labels_list):
    # cajas de longitud variable -> arreglos con relleno (label 0 = relleno)
    k = max(1, max(len(b) for b in boxes_list))
    n = len(boxes_list)
    boxes = np.zeros((n, k, 4), dtype=np.float32)
    labels = np.zeros((n, k), dtype=np.int64)
    for i, (b, l) in enumerate(zip(boxes_list, labels_list)):
        boxes[i, :len(b)] = b
        labels[i, :len(l)] = l
    return boxes, labels


def unpad_boxes(boxes_pad, labels_pad):
    valid = labels_pad > 0
    return boxes_pad[valid].copy(), labels_pad[valid].copy()


def _leer_cache(folder):
    return {
        # las imágenes se leen del disco al vuelo (mmap): con 3 canales la de train pesa varios GB
        "images": np.load(folder / "images.npy", mmap_mode="r"), "inst": np.load(folder / "inst.npy"),
        "boxes": np.load(folder / "boxes.npy"), "labels": np.load(folder / "labels.npy"),
        "meta": pd.read_csv(folder / "meta.csv", dtype={"case_id": str}),
    }


def seleccionar(man, max_slices):
    man = man.sort_values(["case_id", "slice_index"]).reset_index(drop=True)
    return man.iloc[:max_slices].reset_index(drop=True) if max_slices else man


def construir_o_cargar_cache(plane, split, kind, man, max_slices=None):
    """Caché de los cortes `man` (DataFrame con case_id, slice_index, x_mm, mid_true_mm, fov_mm). kind: 'pos' | 'neg'."""
    man = seleccionar(man, max_slices)
    tag = _etiqueta_cache(plane, split, kind, max_slices)
    folder = CACHE_DIR / tag
    key = {**DATA_VERSION, "kind": kind, "split": split, "max_slices": max_slices,
           "cortes": hashlib.sha1(man[["case_id", "slice_index"]].to_csv(index=False).encode()).hexdigest()[:12]}
    key_path = folder / "key.json"
    if key_path.exists() and json.loads(key_path.read_text()) == key:
        log(f"[{tag}] caché cargada: {len(man)} cortes")
        return _leer_cache(folder)

    n, S = len(man), IMAGE_SIZE
    libre = shutil.disk_usage(CACHE_DIR).free
    if n * _bytes_por_corte() > libre - 2 * 1024 ** 3:
        raise RuntimeError(f"[{tag}] no alcanza el disco: la caché necesita ~{n * _bytes_por_corte() / 1e9:.1f} GB y quedan {libre / 1e9:.1f} GB. "
                           f"Liberar espacio, poner CACHE_IMG_DTYPE = 'uint8' o BORRAR_CACHE_AL_TERMINAR = True.")
    folder.mkdir(parents=True, exist_ok=True)
    img_dtype = np.float16 if CACHE_IMG_DTYPE == "float16" else np.uint8
    images = np.lib.format.open_memmap(folder / "images.npy", mode="w+", dtype=img_dtype, shape=(n, N_IMG, S, S))
    inst = np.lib.format.open_memmap(folder / "inst.npy", mode="w+", dtype=np.uint8, shape=(n, S, S))
    boxes_list, labels_list = [], []
    k, t0 = 0, time.perf_counter()
    for case_id, grp in man.groupby("case_id", sort=True):
        row = df[df["case_id"] == case_id].iloc[0]
        vol, sp, _ = load_canonical(row["image_path"], is_label=False, resolver=resolve_repo_path, dtype="native")
        lab, _, _ = load_canonical(row["label_path"], is_label=True, resolver=resolve_repo_path, dtype="native")
        vecinos = {}                                       # imágenes ya preprocesadas de este volumen (los vecinos se reutilizan)
        for rec in grp.itertuples(index=False):
            image, mask, target, _tf = make_sample(
                vol, lab, plane, int(rec.slice_index), sp, window=apply_hu_window, enhance=enhance_bone_contrast,
                letterbox=letterbox_pair, build_target=build_detection_target, case_id=case_id, size=S, min_pixels=MIN_PIXELS,
                n_channels=N_IMG, slice_cache=vecinos)
            image = np.asarray(image, dtype=np.float32).reshape(N_IMG, S, S)
            assert mask.shape == (S, S) and 0 <= mask.min() and mask.max() <= 30, (mask.shape, mask.min(), mask.max())
            if kind == "neg":
                assert len(target["boxes"]) == 0 and mask.max() == 0, f"el corte negativo {case_id}/{rec.slice_index} tiene pelvis"
            images[k] = image.astype(np.float16) if img_dtype == np.float16 else np.clip(np.rint(image * 255.0), 0, 255).astype(np.uint8)
            inst[k] = mask.astype(np.uint8)
            boxes_list.append(np.asarray(target["boxes"], dtype=np.float32).reshape(-1, 4))
            labels_list.append(np.asarray(target["labels"]).astype(np.int64).reshape(-1))
            k += 1
            if k % 2000 == 0 or k == n:
                log(f"[{tag}] {k}/{n} cortes | {time.perf_counter() - t0:.0f} s")
        del vol, lab, vecinos                              # suelta el volumen antes de leer el siguiente
        gc.collect()
    assert k == n, (k, n)
    boxes, labels = pad_boxes(boxes_list, labels_list)
    images.flush(); inst.flush()
    del images, inst
    gc.collect()
    np.save(folder / "boxes.npy", boxes); np.save(folder / "labels.npy", labels)
    man[["case_id", "slice_index", "x_mm", "mid_true_mm", "fov_mm"]].to_csv(folder / "meta.csv", index=False)
    key_path.write_text(json.dumps(key))                   # al final: una construcción interrumpida no queda como válida
    log(f"[{tag}] caché creada: {n} cortes, {man['case_id'].nunique()} casos ({n * _bytes_por_corte() / 1e9:.1f} GB)")
    return _leer_cache(folder)


def borrar_caches(plane):
    for p in CACHE_DIR.glob(f"{plane}_*"):
        if p.is_dir() and ("_pos_" in p.name or "_neg_" in p.name) and not p.name.endswith(tuple(f"_first{k}" for k in range(1000)) ):
            shutil.rmtree(p, ignore_errors=True)
            log(f"caché borrada: {p.name}")

In [6]:
class CachedSliceDataset(torch.utils.data.Dataset):
    # Cortes de la caché. Los targets de segmentación (centros/offsets) se generan al vuelo con la función de Segmentacion_instancias.ipynb.
    def __init__(self, cache, center_mode="interior"):
        self.images, self.inst = cache["images"], cache["inst"]
        self.center_mode = center_mode
        self.boxes, self.labels, self.meta = cache["boxes"], cache["labels"], cache["meta"]
        self.case_ids = self.meta["case_id"].tolist()
        self.slice_ids = self.meta["slice_index"].tolist()
        self.need_seg = True        # False: no se generan centros/offsets/semántica (la etapa A solo entrena detección)

    def __len__(self):
        return len(self.images)

    def _imagen(self, i):
        img = self.images[i].astype(np.float32)
        return img / 255.0 if self.images.dtype == np.uint8 else img

    def __getitem__(self, i):
        inst = self.inst[i].astype(np.int32)
        boxes, labels = unpad_boxes(self.boxes[i], self.labels[i])
        item = {
            "image": torch.from_numpy(self._imagen(i)),
            "instance_mask": torch.from_numpy(inst),
            "boxes": boxes, "labels": labels,
            "case_id": self.case_ids[i], "slice_index": self.slice_ids[i],
        }
        if self.need_seg:
            seg = build_instance_segmentation_targets(inst, center_mode=self.center_mode)
            item.update({"semantic": torch.from_numpy(seg["semantic"]),
                         "centers": torch.from_numpy(seg["centers"]),
                         "offsets": torch.from_numpy(seg["offsets"]),
                         "foreground": torch.from_numpy(seg["foreground"])})
        return item


def collate_slices(batch):
    keys = ("image", "instance_mask") + (("semantic", "centers", "offsets", "foreground") if "semantic" in batch[0] else ())
    out = {k: torch.stack([b[k] for b in batch]) for k in keys}
    out["boxes"] = [b["boxes"] for b in batch]
    out["labels"] = [b["labels"] for b in batch]
    out["case_id"] = [b["case_id"] for b in batch]
    out["slice_index"] = [b["slice_index"] for b in batch]
    return out


def _entero_de_caso(case_id):
    import zlib
    return zlib.crc32(str(case_id).encode())              # estable entre ejecuciones (hash() de Python no lo es)


class PlaneSliceDataset(CachedSliceDataset):
    """Añade (si use_position) un canal constante al final con la posición lateral del corte respecto a la línea media.
    La línea media es UNA por caso y época (misma para sus positivos y sus negativos): en train, real + ruido (o centro del FOV con
    prob. POS_P_FOV), derivada de (semilla, época, caso); en val, fija: 'fov' (centro del campo de visión) o 'true' (centro real del sacro)."""
    def __init__(self, cache, use_position, train, seed=SEED, val_mode="fov", center_mode="interior"):
        super().__init__(cache, center_mode)
        self.use_position, self.train, self.seed, self.val_mode = use_position, train, seed, val_mode
        self.x_mm = self.meta["x_mm"].to_numpy(dtype=np.float64)
        self.mid_true = self.meta["mid_true_mm"].to_numpy(dtype=np.float64)
        self.fov = self.meta["fov_mm"].to_numpy(dtype=np.float64)
        self._case_index = self.meta.groupby("case_id").indices
        self.set_epoch(0)

    def set_epoch(self, epoch):
        self.epoch = int(epoch)
        est = np.empty(len(self.x_mm), dtype=np.float64)
        for case, idx in self._case_index.items():
            mt, fv = float(self.mid_true[idx[0]]), float(self.fov[idx[0]])
            if self.train:
                rng = np.random.default_rng([int(self.seed), self.epoch, _entero_de_caso(case)])
                m = planos.training_midline(mt, fv, rng, POS_SIGMA_MM, POS_P_FOV)
            else:
                m = fv if (self.val_mode == "fov" or not np.isfinite(mt)) else mt
            est[idx] = m
        self.mid_est = est

    def __getitem__(self, i):
        item = super().__getitem__(i)
        if self.use_position:
            value = position_value(self.x_mm[i], self.mid_est[i])
            item["image"] = torch.cat([item["image"], torch.full_like(item["image"][:1], value)], dim=0)    # [N_IMG + 1, H, W]
        return item


class MezclaCortes(torch.utils.data.Dataset):
    """Positivos + un subconjunto de negativos. need_seg y set_epoch se propagan a los dos."""
    def __init__(self, positivos, negativos, indices_negativos):
        self.pos, self.neg = positivos, negativos
        self.idx_neg = [int(i) for i in indices_negativos]

    @property
    def need_seg(self):
        return self.pos.need_seg

    @need_seg.setter
    def need_seg(self, value):
        self.pos.need_seg = self.neg.need_seg = bool(value)

    def set_epoch(self, epoch):
        self.pos.set_epoch(epoch)
        self.neg.set_epoch(epoch)

    def __len__(self):
        return len(self.pos) + len(self.idx_neg)

    def __getitem__(self, i):
        return self.pos[i] if i < len(self.pos) else self.neg[self.idx_neg[i - len(self.pos)]]


VAL_CTX = {"pos": None, "neg": None}      # datasets de val (solo detección) con los que se miden los cortes vacíos; los fija preparar_datos()


def preparar_datos(plane, prueba):
    """Manifiestos -> cachés -> datasets y loaders del plano. Devuelve un dict con loaders y tamaños."""
    global DATA_VERSION
    build_plane_manifests()
    verificar_manifiestos(plane)
    pos, neg = leer_manifiestos(plane)
    DATA_VERSION = {**DATA_VERSION, "manifiesto_pos": _sha1(ruta_manifiesto(plane, "pos")), "manifiesto_neg": _sha1(ruta_manifiesto(plane, "neg"))}

    pos_tr, pos_va = pos[pos["split"] == "train"], pos[pos["split"] == "val"]
    neg_tr, neg_va = neg[neg["split"] == "train"], neg[neg["split"] == "val"]
    if MAX_TRAIN_SLICES:        # prueba corta: solo los negativos de los mismos casos que los positivos elegidos (evita leer decenas de volúmenes)
        neg_tr = neg_tr[neg_tr["case_id"].isin(set(seleccionar(pos_tr, MAX_TRAIN_SLICES)["case_id"]))]
    if MAX_VAL_SLICES:
        neg_va = neg_va[neg_va["case_id"].isin(set(seleccionar(pos_va, MAX_VAL_SLICES)["case_id"]))]
    n_pos_tr = len(seleccionar(pos_tr, MAX_TRAIN_SLICES))
    neg_tr_sel = sample_negatives(neg_tr, int(round(NEG_POR_POSITIVO * n_pos_tr)), seed=SEED, near_share=NEG_CERCANOS) if USE_NEGATIVOS else neg_tr.iloc[:0]

    pedidas = [("train", "pos", pos_tr, MAX_TRAIN_SLICES), ("val", "pos", pos_va, MAX_VAL_SLICES)]
    if USE_NEGATIVOS:
        pedidas += [("train", "neg", neg_tr_sel, MAX_NEG_SLICES), ("val", "neg", neg_va, MAX_NEG_SLICES)]
    falta = sum(len(seleccionar(m, mx)) for s, k, m, mx in pedidas if not (CACHE_DIR / _etiqueta_cache(plane, s, k, mx) / "key.json").exists())
    log(f"{plane}: cortes por construir en caché ~{falta} (~{falta * _bytes_por_corte() / 1e9:.1f} GB) | positivos train {n_pos_tr}, negativos train {len(neg_tr_sel)}")

    cach = {(s, k): construir_o_cargar_cache(plane, s, k, m, mx) for s, k, m, mx in pedidas}
    train_cache, val_cache = cach[("train", "pos")], cach[("val", "pos")]
    neg_train_cache, neg_val_cache = cach.get(("train", "neg")), cach.get(("val", "neg"))
    assert not (set(train_cache["meta"]["case_id"]) & set(val_cache["meta"]["case_id"])), "casos repartidos entre train y val"
    if USE_NEGATIVOS:
        assert (neg_train_cache["labels"] == 0).all() and (neg_val_cache["labels"] == 0).all(), "un negativo trae cajas"
        assert not ((set(train_cache["meta"]["case_id"]) | set(neg_train_cache["meta"]["case_id"]))
                    & (set(val_cache["meta"]["case_id"]) | set(neg_val_cache["meta"]["case_id"]))), "hay casos de val en train"

    def ds(cache, train, val_mode=POS_VAL_MODE):
        return PlaneSliceDataset(cache, USE_POSITION_CHANNEL, train=train, val_mode=val_mode, center_mode=CENTER_MODE)

    pos_ds = ds(train_cache, True)
    if USE_NEGATIVOS:
        neg_ds = ds(neg_train_cache, True)
        train_ds = MezclaCortes(pos_ds, neg_ds, range(len(neg_ds)))
    else:
        train_ds = pos_ds
    gen = torch.Generator()
    gen.manual_seed(SEED)
    train_loader = torch.utils.data.DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, num_workers=NUM_WORKERS,
                                               pin_memory=DEVICE.type == "cuda", collate_fn=collate_slices, generator=gen)
    val_loader = torch.utils.data.DataLoader(ds(val_cache, False), batch_size=VAL_BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
                                             pin_memory=DEVICE.type == "cuda", collate_fn=collate_slices)
    # val para los cortes vacíos: TODOS los positivos y TODOS los negativos de val, solo detección (sin targets de segmentación)
    VAL_CTX["pos"] = ds(val_cache, False)
    VAL_CTX["neg"] = ds(neg_val_cache, False) if USE_NEGATIVOS else None
    for d in (VAL_CTX["pos"], VAL_CTX["neg"]):
        if d is not None:
            d.need_seg = False

    t = time.perf_counter(); b = next(iter(train_loader)); dt = time.perf_counter() - t
    log(f"{plane}: train {len(train_ds)} cortes ({len(pos_ds)} pos + {len(train_ds) - len(pos_ds)} neg), {len(train_loader)} iteraciones/época | "
        f"val {len(val_loader.dataset)} pos" + (f" + {len(VAL_CTX['neg'])} neg" if USE_NEGATIVOS else "") +
        f" | batch {tuple(b['image'].shape)} | carga de un batch {dt * 1000:.0f} ms")
    if USE_POSITION_CHANNEL:
        log(f"{plane}: canal de posición en el primer batch: {[round(float(v), 2) for v in b['image'][:6, -1, 0, 0]]}")
    return {"train_loader": train_loader, "val_loader": val_loader, "n_pos_train": len(pos_ds), "n_neg_train": len(train_ds) - len(pos_ds),
            "n_pos_val": len(val_loader.dataset), "n_neg_val": len(VAL_CTX["neg"]) if USE_NEGATIVOS else 0}

In [7]:
def jitter_boxes(boxes, rng, jitter_px=JITTER_PX):
    # Desplaza cada coordenada hasta jitter_px, sin superar el 20% del lado menor (la caja sigue siendo válida).
    boxes = np.asarray(boxes, dtype=np.float32).reshape(-1, 4)
    if len(boxes) == 0:
        return boxes
    w, h = boxes[:, 2] - boxes[:, 0], boxes[:, 3] - boxes[:, 1]
    j = np.minimum(jitter_px, 0.2 * np.minimum(w, h))[:, None]
    out = boxes + rng.uniform(-1.0, 1.0, size=boxes.shape).astype(np.float32) * j
    return np.clip(out, 0, IMAGE_SIZE).astype(np.float32)


def predicted_boxes_top1(decoded_i, score_min=PRED_BOX_SCORE_MIN):
    # MARCADOR DE POSICIÓN del NMS propio por clase: una caja por clase (la de mayor score).
    boxes, labels = [], []
    for c in (1, 2, 3):
        idx = (decoded_i["labels"] == c).nonzero(as_tuple=True)[0]
        if len(idx) == 0:
            continue
        j = idx[decoded_i["scores"][idx].argmax()]
        if float(decoded_i["scores"][j]) >= score_min:
            boxes.append(decoded_i["boxes"][j].detach().cpu().numpy()); labels.append(c)
    return np.asarray(boxes, dtype=np.float32).reshape(-1, 4), np.asarray(labels, dtype=np.int64)


def make_priors(batch, det_logits, mode, p_pred, rng):
    # mode: "gt" | "gt_jitter" | "pred" | "mixed"
    B = len(batch["boxes"])
    decoded, detections = None, []
    for b in range(B):
        use_pred = mode == "pred" or (mode == "mixed" and rng.random() < p_pred)
        if use_pred:
            if decoded is None:
                decoded = decode_predictions(det_logits.detach().float(), score_thresh=0.0, img_size=IMAGE_SIZE)
            boxes, labels = predicted_boxes_top1(decoded[b])
        elif mode in ("gt_jitter", "mixed"):
            boxes, labels = jitter_boxes(batch["boxes"][b], rng), batch["labels"][b]
        else:
            boxes, labels = batch["boxes"][b], batch["labels"][b]
        detections.append({"boxes": boxes, "labels": labels})
    return build_detection_region_priors(detections, output_size=(IMAGE_SIZE, IMAGE_SIZE), device=DEVICE)


def make_grid_targets(batch):
    tg = [{"boxes": torch.as_tensor(b, dtype=torch.float32), "labels": torch.as_tensor(l, dtype=torch.long)}
          for b, l in zip(batch["boxes"], batch["labels"])]
    return build_targets(tg, grid_s=GRID_SIZE, img_size=IMAGE_SIZE)


def seg_loss_from_batch(seg_out, batch):
    return center_offset_segmentation_loss(
        seg_out, batch["semantic"].to(DEVICE), batch["centers"].to(DEVICE),
        batch["offsets"].to(DEVICE), batch["foreground"].to(DEVICE), weights=SEG_WEIGHTS)


def compute_losses(det_logits, seg_out, batch, grid_t, w_det, w_seg):
    out, total = {}, 0.0
    if w_det > 0:
        det = grid_detection_loss(det_logits.float(), grid_t, lambdas=DET_LAMBDAS)
        out.update({f"det_{k}": v for k, v in det.items()})
        total = total + w_det * det["total"]
    if w_seg > 0:
        seg = seg_loss_from_batch(seg_out, batch)
        out.update({f"seg_{k}": v for k, v in seg.items()})
        total = total + w_seg * seg["total"]
    out["total"] = total
    return out

In [8]:
def _to_np(x):
    return x.detach().cpu().numpy() if torch.is_tensor(x) else np.asarray(x)


def _iou_np(box, boxes):
    boxes = np.asarray(boxes, dtype=np.float64).reshape(-1, 4)
    x1 = np.maximum(box[0], boxes[:, 0]); y1 = np.maximum(box[1], boxes[:, 1])
    x2 = np.minimum(box[2], boxes[:, 2]); y2 = np.minimum(box[3], boxes[:, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area_a = max(box[2] - box[0], 0) * max(box[3] - box[1], 0)
    area_b = np.clip(boxes[:, 2] - boxes[:, 0], 0, None) * np.clip(boxes[:, 3] - boxes[:, 1], 0, None)
    return inter / np.maximum(area_a + area_b - inter, 1e-9)


def _average_precision(decoded, targets, class_id, iou_thr=0.5):
    detections, gts, n_gt = [], {}, 0
    for i, (pred, tgt) in enumerate(zip(decoded, targets)):
        g_boxes = _to_np(tgt["boxes"]).reshape(-1, 4)
        g_labels = _to_np(tgt["labels"]).astype(int).reshape(-1)
        g_boxes = g_boxes[g_labels == class_id]
        gts[i] = (g_boxes, np.zeros(len(g_boxes), dtype=bool))
        n_gt += len(g_boxes)
        p_boxes = _to_np(pred["boxes"]).reshape(-1, 4)
        p_scores = _to_np(pred["scores"]).reshape(-1)
        p_labels = _to_np(pred["labels"]).astype(int).reshape(-1)
        keep = (p_labels == class_id) & (p_scores >= EVAL_SCORE_MIN)
        detections.extend((float(s), i, b) for s, b in zip(p_scores[keep], p_boxes[keep]))
    if n_gt == 0:
        return None
    detections.sort(key=lambda d: -d[0])
    tp, fp = np.zeros(len(detections)), np.zeros(len(detections))
    for k, (_, i, box) in enumerate(detections):
        g_boxes, used = gts[i]
        if len(g_boxes) == 0:
            fp[k] = 1; continue
        ious = _iou_np(box, g_boxes)
        j = int(ious.argmax())
        if ious[j] >= iou_thr and not used[j]:
            used[j] = True; tp[k] = 1
        else:
            fp[k] = 1
    tp, fp = np.cumsum(tp), np.cumsum(fp)
    recall, precision = tp / n_gt, tp / np.maximum(tp + fp, 1e-9)
    mrec = np.concatenate([[0.0], recall, [1.0]]); mpre = np.concatenate([[0.0], precision, [0.0]])
    for k in range(len(mpre) - 2, -1, -1):
        mpre[k] = max(mpre[k], mpre[k + 1])
    idx = np.where(mrec[1:] != mrec[:-1])[0]
    return float(np.sum((mrec[idx + 1] - mrec[idx]) * mpre[idx + 1]))


def detection_metrics(decoded, targets, class_ids=(1, 2, 3)):
    ious = []
    for pred, tgt in zip(decoded, targets):
        p_boxes = _to_np(pred["boxes"]).reshape(-1, 4)
        p_scores = _to_np(pred["scores"]).reshape(-1)
        p_labels = _to_np(pred["labels"]).astype(int).reshape(-1)
        for g_box, g_label in zip(_to_np(tgt["boxes"]).reshape(-1, 4), _to_np(tgt["labels"]).astype(int).reshape(-1)):
            same = np.flatnonzero(p_labels == g_label)
            if len(same) == 0:
                ious.append(0.0); continue
            best = same[p_scores[same].argmax()]
            ious.append(float(_iou_np(g_box, p_boxes[best:best + 1])[0]))
    ious = np.asarray(ious) if ious else np.zeros(1)
    aps = {c: _average_precision(decoded, targets, c) for c in class_ids}
    valid = [a for a in aps.values() if a is not None]
    return {"mean_iou": float(ious.mean()), "recall50": float((ious >= 0.5).mean()),
            "recall75": float((ious >= 0.75).mean()),
            "map50": float(np.mean(valid)) if valid else float("nan"), "ap_by_class": aps}


def fragment_hits(pred_inst, gt_inst, thr=0.5):
    # Fragmentos GT (ids != 0) cuyo mejor IoU con algún fragmento predicho es >= thr.
    gt_ids = [int(i) for i in np.unique(gt_inst) if i != 0]
    pred_masks = [pred_inst == int(p) for p in np.unique(pred_inst) if p != 0]
    hits = 0
    for g in gt_ids:
        gm = gt_inst == g
        best = 0.0
        for pm in pred_masks:
            inter = np.logical_and(gm, pm).sum()
            if inter:
                best = max(best, inter / np.logical_or(gm, pm).sum())
        hits += int(best >= thr)
    return hits, len(gt_ids)

In [9]:
@torch.no_grad()
def evaluate(model, loader, run_seg, instance_n=INSTANCE_EVAL_N):
    model.eval()
    rng = np.random.default_rng(SEED)
    decoded_all, targets_all = [], []
    det_sum, n_img = defaultdict(float), 0
    modes = ("gt", "pred")
    acc = {m: {c: {"inter": 0.0, "psum": 0.0, "gsum": 0.0} for c in (1, 2, 3)} for m in modes}
    inst = {m: {"hit": 0, "total": 0} for m in modes}
    seg_loss_sum, inst_seen = 0.0, 0

    for batch in loader:
        B = len(batch["boxes"])
        x = batch["image"].to(DEVICE, non_blocking=True)
        grid_t = make_grid_targets(batch)
        with autocast():
            feats = model.trunk(x)
            det_logits = model.det_head(feats[-1])
        det = grid_detection_loss(det_logits.float(), grid_t, lambdas=DET_LAMBDAS)
        for k in ("total", "obj", "box", "cls"):
            det_sum[k] += float(det[k]) * B
        n_img += B
        decoded_all += decode_predictions(det_logits.float(), score_thresh=0.0, img_size=IMAGE_SIZE)
        targets_all += [{"boxes": b, "labels": l} for b, l in zip(batch["boxes"], batch["labels"])]
        if not run_seg:
            continue

        gt_sem = batch["semantic"].to(DEVICE)
        take = max(0, min(B, instance_n - inst_seen))
        for mode in modes:
            priors = make_priors(batch, det_logits, mode, 0.0, rng)
            with autocast():
                seg_out = model.segment(x, feats, priors)
            pred = seg_out["semantic_logits"].float().argmax(1)
            for c in (1, 2, 3):
                p, g = pred == c, gt_sem == c
                acc[mode][c]["inter"] += float((p & g).sum()); acc[mode][c]["psum"] += float(p.sum())
                acc[mode][c]["gsum"] += float(g.sum())
            if mode == "gt":
                seg_loss_sum += float(seg_loss_from_batch(seg_out, batch)["total"]) * B
            for i in range(take):
                one = {k: v[i:i + 1].float() for k, v in seg_out.items()}
                pred_inst, _, _ = decode_center_offset_instances(one)
                h, t = fragment_hits(pred_inst, batch["instance_mask"][i].numpy())
                inst[mode]["hit"] += h; inst[mode]["total"] += t
        inst_seen += take

    det_m = detection_metrics(decoded_all, targets_all)
    out = {"val_det_loss": det_sum["total"] / n_img, "val_det_obj": det_sum["obj"] / n_img,
           "val_det_box": det_sum["box"] / n_img, "val_det_cls": det_sum["cls"] / n_img,
           "val_det_mean_iou": det_m["mean_iou"], "val_det_recall50": det_m["recall50"],
           "val_det_recall75": det_m["recall75"], "val_det_map50": det_m["map50"]}
    for c, ap in det_m["ap_by_class"].items():
        out[f"val_det_ap50_c{c}"] = float("nan") if ap is None else ap
    if run_seg:
        out["val_seg_loss_gt"] = seg_loss_sum / n_img
        for mode in modes:
            dices = []
            for c in (1, 2, 3):
                a = acc[mode][c]
                d = 2 * a["inter"] / (a["psum"] + a["gsum"]) if (a["psum"] + a["gsum"]) > 0 else float("nan")
                out[f"val_seg_dice_{mode}_c{c}"] = d
                out[f"val_seg_iou_{mode}_c{c}"] = (a["inter"] / (a["psum"] + a["gsum"] - a["inter"])
                                                  if (a["psum"] + a["gsum"] - a["inter"]) > 0 else float("nan"))
                dices.append(d)
            out[f"val_seg_dice_{mode}"] = float(np.nanmean(dices))
            out[f"val_inst_recall50_{mode}"] = inst[mode]["hit"] / max(inst[mode]["total"], 1)
        out["val_inst_n_gt"] = inst["gt"]["total"]
    return out


def selection_score(kind, v):
    if kind == "det":
        s = v["val_det_map50"]
    elif kind == "seg_gt":
        s = v["val_seg_dice_gt"]
    else:   # "both"
        s = 0.5 * v["val_det_map50"] + 0.5 * v["val_seg_dice_pred"]
    return -math.inf if (s is None or math.isnan(s)) else float(s)

In [10]:
@torch.no_grad()
def scores_dataset(model, ds, batch_size=None):
    """Mejor score tras el NMS de cada corte del dataset (0.0 si no queda caja). Misma regla que cortes_vacios / scores_por_corte."""
    model.eval()
    bs = batch_size or max(VAL_BATCH_SIZE * 4, 16)
    out = []
    for i in range(0, len(ds), bs):
        batch = collate_slices([ds[j] for j in range(i, min(i + bs, len(ds)))])
        x = batch["image"].to(DEVICE)
        with autocast():
            det_logits = model.det_head(model.trunk(x)[-1])
        final = postprocess(det_logits.float(), score_thresh=CONFIG_PIPELINE["score_thresh"], nms_fn=nms_por_clase,
                            iou_thresh=CONFIG_PIPELINE["iou_thresh"], max_det_per_class=CONFIG_PIPELINE["max_det_per_class"])
        out += [float(f["scores"].max()) if f["scores"].numel() else 0.0 for f in final]
    return np.nan_to_num(np.asarray(out, dtype=np.float64), nan=0.0)


def metricas_vacios(model):
    """Resumen de cortes vacíos con el val completo (todos los positivos y todos los negativos). {} si no se usan negativos."""
    if not USE_NEGATIVOS or VAL_CTX["neg"] is None or len(VAL_CTX["neg"]) == 0:
        return {}
    s_pos, s_neg = scores_dataset(model, VAL_CTX["pos"]), scores_dataset(model, VAL_CTX["neg"])
    tiene = np.r_[np.ones(len(s_pos), bool), np.zeros(len(s_neg), bool)]
    r = cv.resumen_cortes_vacios(np.r_[s_pos, s_neg], tiene, recall_min=RECALL_MIN, umbral_fijo=UMBRAL_REFERENCIA)
    return {f"val_{k}": v for k, v in r.items()}


evaluate_base = evaluate
selection_score_base = selection_score


def evaluate(model, loader, run_seg, instance_n=None):
    with torch.no_grad():      # instance_n se lee al llamar (la prueba corta lo cambia en tiempo de ejecución)
        out = evaluate_base(model, loader, run_seg, INSTANCE_EVAL_N if instance_n is None else instance_n)
    out.update(metricas_vacios(model))
    return out


def selection_score(kind, v):
    s = selection_score_base(kind, v)
    if kind == "seg_gt" or not USE_NEGATIVOS or not math.isfinite(s):
        return s
    f = v.get("val_fpr_vacios_cal", float("nan"))
    f = 1.0 if (f is None or math.isnan(f)) else float(f)          # sin dato de cortes vacíos = peor caso
    return (1.0 - SEL_PESO_VACIOS) * s + SEL_PESO_VACIOS * (1.0 - f)

In [11]:
def set_trainable(model, names):
    for n, child in model.named_children():
        for p in child.parameters():
            p.requires_grad = n in names


def set_train_mode(model, names):
    model.train()
    for n, child in model.named_children():
        if n not in names:
            child.eval()


def group_grad_norms(model, names):
    out = {}
    for n in names:
        child = getattr(model, n)
        if child is None:                       # p. ej. cbam=None en la ablación sin CBAM
            continue
        sq = sum(float(p.grad.detach().float().norm()) ** 2 for p in child.parameters() if p.grad is not None)
        out[f"grad_{n}"] = sq ** 0.5
    return out


def trunk_grad_balance(model, batch, rng):
    # Norma del gradiente de cada rama sobre backbone.conv1 (fp32, eval): ¿quién domina el tronco?
    model.eval()
    x = batch["image"].to(DEVICE)
    grid_t = make_grid_targets(batch)
    feats = model.trunk(x)
    det_logits = model.det_head(feats[-1])
    priors = make_priors(batch, det_logits, "gt_jitter", 0.0, rng)
    seg_out = model.segment(x, feats, priors)
    det = grid_detection_loss(det_logits.float(), grid_t, lambdas=DET_LAMBDAS)["total"]
    seg = seg_loss_from_batch(seg_out, batch)["total"]
    w = model.backbone.conv1.weight
    g_det = torch.autograd.grad(det, w, retain_graph=True)[0].norm().item()
    g_seg = torch.autograd.grad(seg, w)[0].norm().item()
    return g_det, g_seg


def current_config():
    cfg = {k: globals()[k] for k in (
        "SEED", "PLANE", "IMAGE_SIZE", "GRID_SIZE", "IN_CHANNELS", "N_IMG_CHANNELS", "USE_POSITION_CHANNEL", "POS_SIGMA_MM", "POS_P_FOV", "POS_VAL_MODE",
        "USE_NEGATIVOS", "NEG_POR_POSITIVO", "NEG_CERCANOS", "SEL_PESO_VACIOS", "RECALL_MIN", "MIN_PIXELS", "CENTER_MODE", "BACKBONE_NAME", "USE_CBAM", "USE_CBAM_GATE", "USE_SKIPS", "BATCH_SIZE", "VAL_BATCH_SIZE",
        "WEIGHT_DECAY", "GRAD_CLIP", "SEG_WEIGHTS", "DET_LAMBDAS_SOURCE", "JITTER_PX", "PRED_BOX_SCORE_MIN",
        "INSTANCE_EVAL_N", "EVAL_SCORE_MIN", "USE_AMP", "SMOKE_TEST", "MAX_TRAIN_SLICES", "MAX_VAL_SLICES", "MAX_NEG_SLICES")}
    cfg["SEG_WEIGHTS"] = list(SEG_WEIGHTS)
    cfg["DET_LAMBDAS"] = dict(DET_LAMBDAS)
    cfg["STAGES"] = STAGES
    cfg["DATA_VERSION"] = DATA_VERSION
    cfg["torch"] = torch.__version__
    cfg["device"] = torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "cpu"
    try:
        cfg["git_commit"] = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True,
                                           cwd=ROOT, timeout=10).stdout.strip() or None
    except Exception:
        cfg["git_commit"] = None
    return cfg


def save_checkpoint(path, model, optimizer, scaler, scheduler, stage, epoch, metrics, config):
    torch.save({"model_state_dict": model.state_dict(), "optimizer_state_dict": optimizer.state_dict(),
                "scaler_state_dict": scaler.state_dict(), "scheduler_state_dict": scheduler.state_dict(),
                "stage": stage["name"], "epoch": epoch,
                "val_metrics": {k: float(v) for k, v in metrics.items()}, "config": config}, path)


def load_weights(model, path):
    state = torch.load(path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(state["model_state_dict"])
    return state


def _config_comparable(cfg):
    return json.loads(json.dumps({k: v for k, v in cfg.items() if k not in ("torch", "device", "git_commit")}, default=str))


def train_stage(model, stage, train_loader, val_loader, config):
    names = set(stage["train"])
    set_trainable(model, names)
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=stage["lr"], weight_decay=WEIGHT_DECAY)
    steps_per_epoch = len(train_loader)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=stage["lr"], total_steps=stage["epochs"] * steps_per_epoch, pct_start=0.1)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    rng = np.random.default_rng(SEED + sum(map(ord, stage["name"])))
    w_det, w_seg = stage["w_det"], stage["w_seg"]
    for _loader in (train_loader, val_loader):          # etapa A: no se calculan centros/offsets (no se usan)
        _loader.dataset.need_seg = w_seg > 0
    need_det = w_det > 0 or stage["prior"] in ("mixed", "pred")
    run_seg_val = w_seg > 0
    best_score, best_path = -math.inf, MODEL_DIR / f"best_{stage['name']}.pth"
    hist_path = OUT_DIR / f"historial_{stage['name']}.csv"
    if _REUSO[0] and hist_path.exists() and best_path.exists():          # retomar: la etapa ya se completó con esta misma configuración
        prev = pd.read_csv(hist_path)
        ck = torch.load(best_path, map_location="cpu", weights_only=False)
        if len(prev) == stage["epochs"] and _config_comparable(ck["config"]) == _config_comparable(config):
            load_weights(model, best_path)
            _epoch_counter[0] += stage["epochs"]
            log(f"=== Etapa {stage['name']}: ya completa con la misma configuración, se reutiliza ({best_path.name}) ===")
            return prev.to_dict("records")
    _REUSO[0] = False            # desde aquí las etapas siguientes se rehacen (dependen de los pesos de esta)
    diag_batch = next(iter(val_loader)) if (w_det > 0 and w_seg > 0) else None
    rows = []
    log(f"=== Etapa {stage['name']} | entrena {sorted(names)} | w_det={w_det} w_seg={w_seg} | "
          f"priors={stage['prior']} | {stage['epochs']} épocas x {steps_per_epoch} iteraciones ===")

    for ep in range(1, stage["epochs"] + 1):
        set_train_mode(model, names)
        _epoch_counter[0] += 1
        train_loader.dataset.set_epoch(_epoch_counter[0])      # línea media del canal de posición: una por caso y época
        p_pred = stage.get("p_pred_end", 0.0) * (ep - 1) / max(stage["epochs"] - 1, 1)
        sums, n_it = defaultdict(float), 0
        gsums, n_g = defaultdict(float), 0
        t_epoch = time.perf_counter(); t_data = 0.0; t_prev = time.perf_counter()
        for it, batch in enumerate(train_loader):
            t_data += time.perf_counter() - t_prev
            x = batch["image"].to(DEVICE, non_blocking=True)
            grid_t = make_grid_targets(batch) if w_det > 0 else None
            optimizer.zero_grad(set_to_none=True)
            with autocast():
                feats = model.trunk(x)
                det_logits = model.det_head(feats[-1]) if need_det else None
                seg_out = None
                if w_seg > 0:
                    priors = make_priors(batch, det_logits, stage["prior"], p_pred, rng)
                    seg_out = model.segment(x, feats, priors)
            losses = compute_losses(det_logits, seg_out, batch, grid_t, w_det, w_seg)
            if not torch.isfinite(losses["total"]):
                raise RuntimeError(f"Pérdida no finita en etapa {stage['name']}, época {ep}, iteración {it}")
            scaler.scale(losses["total"]).backward()
            scaler.unscale_(optimizer)
            gnorm = torch.nn.utils.clip_grad_norm_(params, GRAD_CLIP)
            if it % LOG_GRAD_EVERY == 0:
                for k, v in group_grad_norms(model, names).items():
                    gsums[k] += v
                gsums["grad_total"] += float(gnorm); n_g += 1
            scale_before = scaler.get_scale()
            scaler.step(optimizer); scaler.update()
            if scaler.get_scale() >= scale_before:      # si AMP omitió el paso, no avanzar el scheduler
                scheduler.step()
            for k, v in losses.items():
                sums[k] += float(v.detach()) if torch.is_tensor(v) else float(v)
            n_it += 1
            t_prev = time.perf_counter()

        row = {"stage": stage["name"], "epoch": ep, "lr_end": optimizer.param_groups[0]["lr"], "p_pred": p_pred,
               "time_s": time.perf_counter() - t_epoch, "data_frac": t_data / max(time.perf_counter() - t_epoch, 1e-9)}
        row.update({f"train_{k}": v / n_it for k, v in sums.items()})
        row.update({k: v / max(n_g, 1) for k, v in gsums.items()})
        if getattr(model.cbam, "gamma", None) is not None:
            row["cbam_gamma"] = float(model.cbam.gamma.detach())      # cuánto abrió la compuerta
        if diag_batch is not None:
            g_det, g_seg = trunk_grad_balance(model, diag_batch, rng)
            row.update({"trunk_grad_det": g_det, "trunk_grad_seg": g_seg})
        val = evaluate(model, val_loader, run_seg=run_seg_val)
        row.update(val)
        score = selection_score(stage["select"], val)
        row["select_score"] = score
        if score > best_score:
            best_score = score
            save_checkpoint(best_path, model, optimizer, scaler, scheduler, stage, ep, val, config)
            row["is_best"] = True
        save_checkpoint(MODEL_DIR / f"last_{stage['name']}.pth", model, optimizer, scaler, scheduler, stage, ep, val, config)
        rows.append(row)
        pd.DataFrame(rows).to_csv(OUT_DIR / f"historial_{stage['name']}.csv", index=False)

        msg = (f"[{stage['name']}] ép {ep:02d}/{stage['epochs']} | train total={row['train_total']:.4f} | "
               f"val mAP50={val['val_det_map50']:.3f} IoU={val['val_det_mean_iou']:.3f}")
        if run_seg_val:
            msg += (f" | Dice gt={val['val_seg_dice_gt']:.3f} pred={val['val_seg_dice_pred']:.3f}"
                    f" | frag@.5 gt={val['val_inst_recall50_gt']:.2f} pred={val['val_inst_recall50_pred']:.2f}")
        if "val_fpr_vacios_cal" in val:
            msg += (f" | vacíos que pasan={val['val_fpr_vacios_cal']:.3f} (umbral {val['val_umbral_cal']:.2f}, recall pelvis {val['val_recall_pelvis_cal']:.3f})")
        msg += f" | sel={score:.4f} | {row['time_s']:.0f}s (datos {row['data_frac']:.0%})" + (" *" if row.get("is_best") else "")
        log(msg)

    for _loader in (train_loader, val_loader):
        _loader.dataset.need_seg = True
    load_weights(model, best_path)      # la siguiente etapa parte del mejor, no del último
    log(f"Etapa {stage['name']}: mejor score {best_score:.4f} -> {best_path.name}")
    return rows

In [12]:
def export_inference_checkpoint(src_path, dst_path):
    """Convierte un checkpoint de entrenamiento (con optimizador, scaler...) en un archivo ligero solo para inferencia."""
    state = torch.load(src_path, map_location="cpu", weights_only=False)
    cfg = state["config"]
    n_img = int(cfg.get("N_IMG_CHANNELS", 1))
    usa_pos = bool(cfg.get("USE_POSITION_CHANNEL", False))
    payload = {
        "format_version": 1,
        "model_state_dict": {k: v.cpu() for k, v in state["model_state_dict"].items()},
        "architecture": {
            "class": "MultiTaskNet", "backbone_name": cfg["BACKBONE_NAME"], "use_cbam": cfg["USE_CBAM"],
            "use_gate": cfg.get("USE_CBAM_GATE", True), "use_skips": cfg.get("USE_SKIPS", True),
            "in_channels": int(cfg.get("IN_CHANNELS", 1)), "image_channels": n_img,
            "plane": cfg.get("PLANE", "axial"),
            "position": {"enabled": usa_pos, "scale_mm": planos.POSITION_SCALE_MM, "clip": planos.POSITION_CLIP, "channel_index": n_img if usa_pos else None,
                         "positive_is": "izquierda del paciente (LPS)"},
            "center_mode": cfg.get("CENTER_MODE", "mean"), "image_size": cfg["IMAGE_SIZE"], "grid_size": cfg["GRID_SIZE"],
        },
        # Valores por defecto de la inferencia: AJUSTAR con val antes de usarlos como definitivos.
        "inference_defaults": {"center_threshold": 0.25, "min_center_distance": 7, "min_instance_pixels": 1,
                               "pred_box_score_min": cfg["PRED_BOX_SCORE_MIN"]},
        "class_names": {int(k): str(v) for k, v in CLASS_NAMES.items()},
        "selected_from": {"stage": state["stage"], "epoch": int(state["epoch"]), "val_metrics": state["val_metrics"]},
        "training_config": json.loads(json.dumps(cfg, default=str)),
        "torch_version": str(torch.__version__),
    }
    torch.save(payload, dst_path)
    return payload


def load_trained_model(path, device=DEVICE):
    """Reconstruye el modelo desde el archivo exportado y lo deja en eval()."""
    payload = torch.load(path, map_location=device, weights_only=True)
    arch = payload["architecture"]
    model = MultiTaskNet(arch["backbone_name"], arch["use_cbam"], arch.get("use_gate", True), arch.get("use_skips", True),
                         arch.get("in_channels", 1), arch.get("image_channels", arch.get("in_channels", 1))).to(device)
    model.load_state_dict(payload["model_state_dict"])
    model.eval()
    return model, payload


def exportar_y_comprobar(val_loader, destino):
    payload = export_inference_checkpoint(MODEL_DIR / "best_C_conjunto.pth", destino)
    log(f"[{PLANE}] guardado: {destino} ({destino.stat().st_size / 1e6:.1f} MB) | salió de {payload['selected_from']['stage']} época {payload['selected_from']['epoch']}")
    reloaded, meta = load_trained_model(destino)
    arch = meta["architecture"]
    reference = MultiTaskNet(arch["backbone_name"], arch["use_cbam"], arch.get("use_gate", True), arch.get("use_skips", True),
                             arch["in_channels"], arch["image_channels"]).to(DEVICE)
    load_weights(reference, MODEL_DIR / "best_C_conjunto.pth")
    reference.eval()
    x = next(iter(val_loader))["image"][:4].to(DEVICE)
    with torch.no_grad():
        f_ref, f_new = reference.trunk(x), reloaded.trunk(x)
        d_ref, d_new = reference.det_head(f_ref[-1]), reloaded.det_head(f_new[-1])
    assert all(torch.allclose(a, b, atol=1e-6) for a, b in zip(f_ref, f_new)) and torch.allclose(d_ref, d_new, atol=1e-6), "el modelo reconstruido no coincide"
    log(f"[{PLANE}] comprobación OK: el modelo reconstruido coincide con el checkpoint")
    return reloaded, payload


def graficar_historial(history, path):
    fig, axes = plt.subplots(1, 4, figsize=(21, 4.2))
    x = np.arange(len(history))
    for key, label in [("train_det_total", "detección"), ("train_seg_total", "segmentación"), ("train_total", "total")]:
        if key in history:
            axes[0].plot(x, history[key], label=label)
    axes[0].set(title="Pérdida de entrenamiento", xlabel="época (acumulada)"); axes[0].legend(); axes[0].set_yscale("log")
    for key, label in [("val_det_map50", "mAP@0.5 detección"), ("val_det_mean_iou", "IoU detección")]:
        axes[1].plot(x, history[key], label=label)
    axes[1].set(title="Validación: detección", xlabel="época (acumulada)", ylim=(0, 1)); axes[1].legend()
    for key, label in [("val_seg_dice_gt", "Dice (cajas reales)"), ("val_seg_dice_pred", "Dice (cajas predichas)")]:
        if key in history:
            axes[2].plot(x, history[key], label=label)
    axes[2].set(title="Validación: segmentación semántica", xlabel="época (acumulada)", ylim=(0, 1)); axes[2].legend()
    for key, label in [("val_fpr_vacios_cal", "vacíos que pasan (umbral recalibrado)"), ("val_vacios_con_caja", "vacíos con alguna caja"),
                       ("val_recall_pelvis_cal", "recall de pelvis (recalibrado)")]:
        if key in history:
            axes[3].plot(x, history[key], label=label)
    axes[3].set(title="Validación: cortes vacíos", xlabel="época (acumulada)", ylim=(0, 1)); axes[3].legend(fontsize=8)
    for ax in axes:
        ax.grid(alpha=0.25)
        for b in np.flatnonzero(history["stage"].ne(history["stage"].shift()).to_numpy())[1:]:
            ax.axvline(b - 0.5, color="gray", ls="--", lw=0.8)
    fig.suptitle(f"{PLANE}", y=1.02)
    plt.tight_layout()
    fig.savefig(path, dpi=130, bbox_inches="tight")
    plt.close(fig)


def graficar_scores_vacios(model, path):
    """Distribución del mejor score por corte (positivos vs vacíos de val) y el umbral recalibrado."""
    if not USE_NEGATIVOS or VAL_CTX["neg"] is None or len(VAL_CTX["neg"]) == 0:
        return {}
    s_pos, s_neg = scores_dataset(model, VAL_CTX["pos"]), scores_dataset(model, VAL_CTX["neg"])
    tiene = np.r_[np.ones(len(s_pos), bool), np.zeros(len(s_neg), bool)]
    u, _ = cv.calibrar_umbral(np.r_[s_pos, s_neg], tiene, recall_min=RECALL_MIN)
    fig, ax = plt.subplots(figsize=(7, 4))
    bins = np.linspace(0, 1, 41)
    ax.hist(s_pos, bins=bins, alpha=0.6, density=True, label=f"con pelvis ({len(s_pos)})")
    ax.hist(s_neg, bins=bins, alpha=0.6, density=True, label=f"sin pelvis ({len(s_neg)})")
    ax.axvline(u, color="k", ls="--", label=f"umbral recalibrado {u:.2f}")
    ax.set(title=f"{PLANE}: mejor score por corte (val)", xlabel="score", ylabel="densidad"); ax.legend(); ax.grid(alpha=0.25)
    fig.savefig(path, dpi=130, bbox_inches="tight")
    plt.close(fig)
    return {"umbral_cal": u}


def metricas_finales(model, val_loader):
    """Métricas del modelo exportado sobre todo el val (positivos), más cortes vacíos. En sagital, también con la línea media REAL del sacro."""
    out = {"val": evaluate(model, val_loader, run_seg=True, instance_n=INSTANCE_EVAL_N)}
    if USE_POSITION_CHANNEL:
        ds_true = PlaneSliceDataset({"images": val_loader.dataset.images, "inst": val_loader.dataset.inst, "boxes": val_loader.dataset.boxes,
                                     "labels": val_loader.dataset.labels, "meta": val_loader.dataset.meta}, True, train=False, val_mode="true",
                                    center_mode=CENTER_MODE)
        dl = torch.utils.data.DataLoader(ds_true, batch_size=VAL_BATCH_SIZE, shuffle=False, num_workers=0, collate_fn=collate_slices)
        with torch.no_grad():
            out["val_linea_media_real"] = evaluate_base(model, dl, True, INSTANCE_EVAL_N)
    return out

In [13]:
# ---------------------------------------------------------------------------
# Visualizar detección + segmentación (semántica e instancias) en train y val.
# Pegar en una celda del cuaderno de entrenamiento, DESPUES de entrenar.
# Usa lo que ya existe alli: torch, np, plt, OUT_DIR, DEVICE, IMAGE_SIZE, autocast,
# collate_slices, decode_predictions, build_detection_region_priors,
# decode_center_offset_instances, train_loader, val_loader y el modelo `model`
# (tras entrenar queda con los mejores pesos de la etapa C).
#
# Columnas:  1) cajas reales (verde) vs predichas (roja) | 2) semántica real
#            3) semántica predicha | 4) instancias reales | 5) instancias predichas
# Color = región anatómica; el tono distinto separa fragmentos de la misma región.
# ---------------------------------------------------------------------------
from matplotlib.patches import Rectangle

REGION_NAMES = {1: "sacro", 2: "coxal izq.", 3: "coxal der."}
REGION_RGB = {1: (0.95, 0.60, 0.10), 2: (0.20, 0.45, 0.95), 3: (0.15, 0.75, 0.40)}
SHADES = [0.0, 0.5, 0.25, 0.7, 0.35, 0.8]


def _id_to_class(i):                         # mismo convenio de IDs que la sección 23
    return 1 if i <= 10 else 2 if i <= 20 else 3


def _semantic_rgba(sem, alpha=0.6):
    rgba = np.zeros((*sem.shape, 4), dtype=np.float32)
    for c, rgb in REGION_RGB.items():
        m = sem == c
        rgba[m, :3] = rgb
        rgba[m, 3] = alpha
    return rgba


def _instance_rgba(inst, id_to_class, alpha=0.7):
    rgba = np.zeros((*inst.shape, 4), dtype=np.float32)
    per_class = defaultdict(list)
    for i in np.unique(inst):
        if i != 0:
            per_class[id_to_class[int(i)]].append(int(i))
    for c, ids in per_class.items():
        for j, i in enumerate(sorted(ids)):
            s = SHADES[j % len(SHADES)]
            m = inst == i
            rgba[m, :3] = np.array(REGION_RGB[c]) * (1 - s) + s
            rgba[m, 3] = alpha
    return rgba


def _draw_contours(ax, inst):
    for i in np.unique(inst):
        if i != 0:
            ax.contour(inst == i, levels=[0.5], colors="k", linewidths=0.5)


def _draw_boxes(ax, boxes, labels, color, scores=None, label=True):
    for k, (b, c) in enumerate(zip(boxes, labels)):
        ax.add_patch(Rectangle((b[0], b[1]), b[2] - b[0], b[3] - b[1], fill=False, ec=color, lw=1.6))
        if label:
            txt = REGION_NAMES.get(int(c), str(c)) + (f" {scores[k]:.2f}" if scores is not None else "")
            ax.text(b[0], max(b[1] - 2, 7), txt, color=color, fontsize=6.5, va="bottom")


def _top1_with_scores(decoded_i, score_min=PRED_BOX_SCORE_MIN):
    # Misma regla que predicted_boxes_top1 (marcador de posición del NMS), pero devolviendo el score.
    boxes, labels, scores = [], [], []
    for c in (1, 2, 3):
        idx = (decoded_i["labels"] == c).nonzero(as_tuple=True)[0]
        if len(idx) == 0:
            continue
        j = idx[decoded_i["scores"][idx].argmax()]
        if float(decoded_i["scores"][j]) >= score_min:
            boxes.append(decoded_i["boxes"][j].detach().cpu().numpy())
            labels.append(c)
            scores.append(float(decoded_i["scores"][j]))
    return np.asarray(boxes, np.float32).reshape(-1, 4), np.asarray(labels, np.int64), np.asarray(scores, np.float32)


def _slice_dice(pred, gt):
    ds = []
    for c in (1, 2, 3):
        g, p = gt == c, pred == c
        if g.any():
            ds.append(2 * (g & p).sum() / (g.sum() + p.sum()))
    return float(np.mean(ds)) if ds else float("nan")


def _pick_indices(ds, n, mode, rng):
    if mode == "spread":                      # repartidos por todo el split (varios pacientes)
        return np.linspace(0, len(ds) - 1, n).astype(int)
    # "fragments": cortes con más fragmentos, de casos distintos, dentro de un grupo aleatorio
    pool = rng.choice(len(ds), size=min(1500, len(ds)), replace=False)
    counts = np.array([len(np.unique(ds.inst[i])) - 1 for i in pool])
    order = np.argsort(-counts)
    chosen, seen = [], set()
    for k in order:
        if ds.case_ids[pool[k]] not in seen:
            chosen.append(int(pool[k])); seen.add(ds.case_ids[pool[k]])
        if len(chosen) == n:
            break
    for k in order:                           # si hay menos casos que n, completar
        if len(chosen) == n:
            break
        if int(pool[k]) not in chosen:
            chosen.append(int(pool[k]))
    return np.array(chosen)


@torch.no_grad()
def visualize_predictions(model, dataset, split_name, n=4, mode="spread", seed=0,
                          center_threshold=0.25, min_center_distance=7, min_instance_pixels=1):
    rng = np.random.default_rng(seed)
    idxs = _pick_indices(dataset, n, mode, rng)
    batch = collate_slices([dataset[int(i)] for i in idxs])
    model.eval()
    with autocast():
        x = batch["image"].to(DEVICE)
        feats = model.trunk(x)
        det_logits = model.det_head(feats[-1])
    decoded = decode_predictions(det_logits.float(), score_thresh=0.0, img_size=IMAGE_SIZE)
    tops = [_top1_with_scores(d) for d in decoded]
    priors = build_detection_region_priors([{"boxes": b, "labels": l} for b, l, _ in tops],
                                           output_size=(IMAGE_SIZE, IMAGE_SIZE), device=DEVICE)
    with autocast():
        seg_out = model.segment(x, feats, priors)          # el decoder usa las cajas PREDICHAS

    titles = ["cajas: real (verde) vs predicha (roja)", "semántica real", "semántica predicha",
              "instancias reales", "instancias predichas"]
    fig, axes = plt.subplots(n, 5, figsize=(15.5, 3.2 * n))
    axes = np.atleast_2d(axes)
    for r in range(n):
        img = batch["image"][r, N_IMG // 2].numpy()      # canal central = corte z (con 2.5D, 0 es z-1)
        gt_sem, gt_inst = batch["semantic"][r].numpy(), batch["instance_mask"][r].numpy()
        one = {k: v[r:r + 1].float() for k, v in seg_out.items()}
        pred_inst, pred_sem, pred_cls = decode_center_offset_instances(
            one, center_threshold, min_center_distance, min_instance_pixels)
        pb, pl, ps = tops[r]
        n_gt = int((np.unique(gt_inst) != 0).sum())
        gt_cls = {int(i): _id_to_class(int(i)) for i in np.unique(gt_inst) if i != 0}

        panels = [None, _semantic_rgba(gt_sem), _semantic_rgba(pred_sem),
                  _instance_rgba(gt_inst, gt_cls), _instance_rgba(pred_inst, pred_cls)]
        for j, ax in enumerate(axes[r]):
            ax.imshow(img, cmap="gray", vmin=0, vmax=1, origin=_ORIGIN)
            if panels[j] is not None:
                ax.imshow(panels[j], interpolation="nearest", origin=_ORIGIN)
            ax.axis("off")
            if r == 0:
                ax.set_title(titles[j], fontsize=9)
        _draw_boxes(axes[r, 0], batch["boxes"][r], batch["labels"][r], "lime", label=False)
        _draw_boxes(axes[r, 0], pb, pl, "red", ps)
        _draw_contours(axes[r, 3], gt_inst)
        _draw_contours(axes[r, 4], pred_inst)
        axes[r, 0].text(2, 252, f"{split_name} · caso {batch['case_id'][r]} · corte {batch['slice_index'][r]}",
                        color="yellow", fontsize=7, va="bottom")
        axes[r, 2].text(2, 252, f"Dice {_slice_dice(pred_sem, gt_sem):.2f}", color="yellow", fontsize=8, va="bottom")
        axes[r, 3].text(2, 252, f"{n_gt} fragmentos", color="yellow", fontsize=8, va="bottom")
        axes[r, 4].text(2, 252, f"{len(pred_cls)} fragmentos", color="yellow", fontsize=8, va="bottom")
    plt.tight_layout()
    out = FIG_DIR / f"vis_{split_name}_{mode}.png"
    fig.savefig(out, dpi=130, bbox_inches="tight")
    plt.close(fig)
    log(f"[{PLANE}] figura guardada: {out}")


def revision_visual(model, train_loader, val_loader):
    pos_train = getattr(train_loader.dataset, "pos", train_loader.dataset)        # con negativos, el dataset de train es una mezcla
    for split_name, ds in (("train", pos_train), ("val", val_loader.dataset)):
        visualize_predictions(model, ds, split_name, n=4, mode="spread")
        visualize_predictions(model, ds, split_name, n=4, mode="fragments")

In [14]:
_epoch_counter = [0]
_REUSO = [True]
_PLANOS_OK = {}


def configurar_plano(plane, prueba):
    g = globals()
    assert plane in PLANES, plane
    g["PLANE"], g["SMOKE_TEST"] = plane, bool(prueba)
    g["RESUME"] = (not FORCE_RETRAIN) and not prueba
    g["USE_POSITION_CHANNEL"] = plane == "sagital"
    g["N_IMG"] = 3 if USE_25D else 1
    g["N_IMG_CHANNELS"] = N_IMG
    g["IN_CHANNELS"] = N_IMG + (1 if USE_POSITION_CHANNEL else 0)
    for k, v in _BASE.items():
        g[k] = v
    g["MAX_TRAIN_SLICES"] = g["MAX_VAL_SLICES"] = g["MAX_NEG_SLICES"] = None
    g["STAGES"] = copy.deepcopy(STAGES_BASE)
    if prueba:
        for k in ("MAX_TRAIN_SLICES", "MAX_VAL_SLICES", "MAX_NEG_SLICES", "INSTANCE_EVAL_N", "BATCH_SIZE", "VAL_BATCH_SIZE"):
            g[k] = AJUSTES_PRUEBA[k]
        for st in g["STAGES"]:
            st["epochs"] = AJUSTES_PRUEBA["epochs"]
    g["ETAPA"] = f"entrenamiento_multitarea_{plane}" + ("_smoke" if prueba else "")
    g["OUT_DIR"], g["MODEL_DIR"], g["FIG_DIR"] = METRICAS_DIR / ETAPA, MODELOS_DIR / ETAPA, FIGURAS_DIR / ETAPA
    if prueba:
        for d in (OUT_DIR, MODEL_DIR, FIG_DIR):
            shutil.rmtree(d, ignore_errors=True)
    for d in (OUT_DIR, MODEL_DIR, FIG_DIR):
        d.mkdir(parents=True, exist_ok=True)
    g["_ORIGIN"] = "upper" if plane == "axial" else "lower"
    _epoch_counter[0] = 0
    _REUSO[0] = RESUME
    g["DATA_VERSION"] = {"version": "planos-25d-neg-1", "plane": plane, "n_img": N_IMG, "dtype": CACHE_IMG_DTYPE, "min_pixels": MIN_PIXELS,
                         "image_size": IMAGE_SIZE, "dataset_csv": _sha1(DATASET_CSV),
                         "planos_py": _sha1(planos.__file__), "orientacion_py": _sha1(orientacion.__file__)}


def _resumen_de(plane, history, finales, payload):
    c = history[history["stage"] == "C_conjunto"]
    mejor = c.loc[c["select_score"].idxmax()] if len(c) else history.iloc[-1]
    v = finales["val"]
    fila = {"plano": plane, "estado": "ok", "epoca_elegida": int(mejor["epoch"]), "mAP50": v["val_det_map50"], "Dice_pred": v["val_seg_dice_pred"],
            "Dice_gt": v["val_seg_dice_gt"], "frag_recall50_pred": v["val_inst_recall50_pred"], "vacios_que_pasan_cal": v.get("val_fpr_vacios_cal"),
            "umbral_cal": v.get("val_umbral_cal"), "recall_pelvis_cal": v.get("val_recall_pelvis_cal"),
            "vacios_con_caja": v.get("val_vacios_con_caja"), "modelo": str(MODEL_DIR / f"modelo_multitarea_{plane}.pt")}
    if "val_linea_media_real" in finales:
        fila["mAP50_linea_real"] = finales["val_linea_media_real"]["val_det_map50"]
        fila["Dice_pred_linea_real"] = finales["val_linea_media_real"]["val_seg_dice_pred"]
    return fila


def entrenar_plano(plane, prueba):
    """Todo el flujo de un plano. Devuelve la fila de resumen."""
    configurar_plano(plane, prueba)
    destino = MODEL_DIR / f"modelo_multitarea_{plane}.pt"
    log(f"{'=' * 20} PLANO {plane.upper()} ({'prueba corta' if prueba else 'completo'}) {'=' * 20}")
    if not prueba and RESUME and destino.exists() and (OUT_DIR / "resumen_plano.json").exists():
        fila = json.loads((OUT_DIR / "resumen_plano.json").read_text())
        log(f"[{plane}] ya estaba entrenado y exportado: se omite ({destino}). FORCE_RETRAIN = True para rehacerlo.")
        fila["estado"] = "ya existía"
        return fila
    t0 = time.perf_counter()
    datos = preparar_datos(plane, prueba)
    train_loader, val_loader = datos["train_loader"], datos["val_loader"]
    seed_everything(SEED)
    model = construir_modelo(verbose=not prueba)
    CONFIG = current_config()
    (OUT_DIR / "config.json").write_text(json.dumps(CONFIG, indent=2, default=str), encoding="utf-8")
    all_rows = []
    for stage in STAGES:
        all_rows += train_stage(model, stage, train_loader, val_loader, CONFIG)
    history = pd.DataFrame(all_rows)
    history.to_csv(OUT_DIR / "historial_completo.csv", index=False)
    graficar_historial(history, FIG_DIR / "historial.png")
    reloaded, payload = exportar_y_comprobar(val_loader, destino)
    graficar_scores_vacios(reloaded, FIG_DIR / "scores_vacios.png")
    finales = metricas_finales(reloaded, val_loader)
    (OUT_DIR / "metricas_val_final.json").write_text(json.dumps(finales, indent=2, default=float), encoding="utf-8")
    try:
        revision_visual(reloaded, train_loader, val_loader)
    except Exception:
        log(f"[{plane}] AVISO: la revisión visual falló (el modelo ya está exportado):\n{traceback.format_exc()}")
    fila = _resumen_de(plane, history, finales, payload)
    fila["minutos"] = round((time.perf_counter() - t0) / 60, 1)
    if not prueba:
        (OUT_DIR / "resumen_plano.json").write_text(json.dumps(fila, indent=2, default=float), encoding="utf-8")
    log(f"[{plane}] listo en {fila['minutos']} min | mAP50 {fila['mAP50']:.3f} | Dice(pred) {fila['Dice_pred']:.3f} | "
        f"vacíos que pasan {fila['vacios_que_pasan_cal']}")
    if BORRAR_CACHE_AL_TERMINAR and not prueba:
        borrar_caches(plane)
    del model, reloaded, datos, train_loader, val_loader
    liberar_memoria()
    return fila


def liberar_memoria():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [15]:
def verificaciones_previas():
    """Comprobaciones baratas (segundos) antes de gastar horas. Lanza AssertionError si algo no cuadra."""
    # 1) la regla de vecinos de planos.py es la del equipo (src/entrada_25d.py)
    for n in (1, 2, 5, 40):
        for z in range(n):
            for c in (1, 3):
                assert neighbor_indices(z, n, c) == indices_vecinos(z, n, c), (z, n, c)
    log("verificación: neighbor_indices == indices_vecinos (src/entrada_25d.py)")
    # 2) el espacio en disco alcanza para las cachés de los planos pedidos
    libre = shutil.disk_usage(CACHE_DIR).free
    log(f"disco libre en {CACHE_DIR}: {libre / 1e9:.0f} GB")
    # 3) los splits no se cruzan
    s = df.groupby("split")["case_id"].apply(set)
    assert not (s.get("train", set()) & s.get("val", set())) and not (s.get("train", set()) & s.get("test", set())), "fuga entre splits"
    # 4) coincidencia de los manifiestos axiales con los del equipo (si existen): mismas filas de positivos
    if slice_manifest is not None and ruta_manifiesto("axial", "pos").exists():
        a = set(map(tuple, pd.read_csv(ruta_manifiesto("axial", "pos"), dtype={"case_id": str})[["case_id", "slice_index"]].to_numpy()))
        col = "slice_index" if "slice_index" in slice_manifest.columns else None
        if col:
            b = set(map(tuple, slice_manifest[["case_id", col]].to_numpy()))
            log(f"verificación: positivos axiales propios {len(a)} vs del equipo {len(b)} | coinciden {len(a & b)} | solo propios {len(a - b)} | solo del equipo {len(b - a)}")

## 2. Fusión 3D, diagnóstico y distancias (solo definiciones; no corren nada)

In [16]:
# ---------------------------------------------------------------------------
# FUSIÓN 3D DE LOS TRES PLANOS (validación).  Pegar en una celda NUEVA al final de Entrenamiento_multitarea_planos.ipynb
# y ejecutar en el mismo kernel (con las definiciones ya cargadas; no reentrena nada).
#
# Qué hace, por cada paciente del split:
#   1. Infiere el VOLUMEN COMPLETO con cada modelo (axial, coronal, sagital) en modo producción: cajas predichas -> priors -> semántica.
#      Usa la misma entrada que el entrenamiento (2.5D z-1,z,z+1; sagital con el canal de posición a partir del centro del campo de visión).
#   2. Devuelve cada probabilidad de corte (256x256) a la forma original del corte y reensambla un volumen de probabilidades
#      (3 clases, uint8) en la orientación canónica. Se guarda en disco: la inferencia es lo único lento y no se repite.
#   3. Fusiona: promedio ponderado de probabilidades -> argmax.   Variantes: cada plano solo, pares y los 3 (con distintos pesos).
#   4. Mide en 3D contra la etiqueta real: Dice por clase y por paciente, cortes axiales vacíos con predicción, y fragmentos 3D
#      (componentes conexos de cada clase; acierto = IoU >= 0.5 con un fragmento real de la misma región).
# El split "test" NO se toca salvo que se pida con confirmar_test=True (una sola vez, al final).
# ---------------------------------------------------------------------------
from PIL import Image                                   # restaura el `Image` de PIL que necesitan las funciones del equipo
from IPython.display import display
from scipy import ndimage as ndi
from scipy.optimize import linear_sum_assignment
import matplotlib.pyplot as plt

FUSION_DIR = MODELOS_DIR.parent / "fusion_3d"            # resultados/fusion_3d
PROB_DIR = FUSION_DIR / "probs"
PROB_DIR.mkdir(parents=True, exist_ok=True)
FUSION_BATCH = 24                                        # cortes por pasada por la red
MIN_MM3_GRID = (0, 100, 500)                             # fragmentos 3D: tamaños mínimos probados (mm³); 0 = no filtrar
MIN_MM3_LIMPIEZA = 100                                   # tamaño mínimo con el que se calcula el "Dice limpio"
VARIANTES = {                                            # nombre -> peso de cada plano en el promedio de probabilidades
    "axial": {"axial": 1},
    "coronal": {"coronal": 1},
    "sagital": {"sagital": 1},
    "axial+coronal": {"axial": 1, "coronal": 1},
    "axial+sagital": {"axial": 1, "sagital": 1},
    "coronal+sagital": {"coronal": 1, "sagital": 1},
    "3 planos (iguales)": {"axial": 1, "coronal": 1, "sagital": 1},
    "3 planos (axial x2)": {"axial": 2, "coronal": 1, "sagital": 1},
}
VARIANTES_FRAG = ("axial", "coronal", "sagital", "3 planos (iguales)")      # a cuáles se les cuentan los fragmentos 3D (cuesta ~1 min por variante y caso)
_STRUCT26 = np.ones((3, 3, 3), dtype=bool)
CLASES = {1: "sacro", 2: "coxal izq.", 3: "coxal der."}
RES_FUSION = {}


# ------------------------------------------------------------------------------------------ inferencia por volumen
def ruta_modelo(plane):
    return MODELOS_DIR / f"entrenamiento_multitarea_{plane}" / f"modelo_multitarea_{plane}.pt"


def _como_en_cache(img):
    """Cuantiza la imagen igual que la caché de entrenamiento (float16 o uint8), para que la red vea lo mismo que vio al entrenar."""
    if CACHE_IMG_DTYPE == "float16":
        return img.astype(np.float16).astype(np.float32)
    return np.clip(np.rint(img * 255.0), 0, 255).astype(np.uint8).astype(np.float32) / 255.0


@torch.no_grad()
def _forward_lote(model, x_np, rng):
    """x_np: [B, C, 256, 256] -> probabilidades de las 3 clases de pelvis [B, 3, 256, 256] en uint8 (el fondo es 255 - suma)."""
    x = torch.from_numpy(x_np).to(DEVICE)
    with autocast():
        feats = model.trunk(x)
        det_logits = model.det_head(feats[-1])
    priors = make_priors({"boxes": [None] * len(x_np), "labels": [None] * len(x_np)}, det_logits, "pred", 0.0, rng)
    with autocast():
        seg_out = model.segment(x, feats, priors)
    p = torch.softmax(seg_out["semantic_logits"].float(), dim=1)[:, 1:]
    return (p * 255.0).round().clamp(0, 255).to(torch.uint8).cpu().numpy()


def _restaurar_probs(p, square_shape, orig_shape, size=256):
    """[3, 256, 256] uint8 -> [3, *orig_shape]: quita el relleno del letterbox y remuestrea (bilineal) a la forma original del corte."""
    h, w = square_shape
    scale = min(size / w, size / h)
    nw, nh = max(1, int(round(w * scale))), max(1, int(round(h * scale)))
    ox, oy = (size - nw) // 2, (size - nh) // 2
    out = np.empty((p.shape[0],) + tuple(orig_shape), dtype=np.uint8)
    for c in range(p.shape[0]):
        crop = np.ascontiguousarray(p[c, oy:oy + nh, ox:ox + nw], dtype=np.float32)
        big = np.asarray(Image.fromarray(crop).resize((int(orig_shape[1]), int(orig_shape[0])), Image.Resampling.BILINEAR), dtype=np.float32)
        out[c] = np.clip(np.rint(big), 0, 255).astype(np.uint8)
    return out


def inferir_volumen(model, plane, vol, sp, batch_size=None):
    """Volumen canónico (z,y,x) -> probabilidades uint8 [3, z, y, x] de sacro / coxal izq. / coxal der. con el modelo de ese plano.
    Requiere configurar_plano(plane, False) (N_IMG y USE_POSITION_CHANNEL del plano)."""
    batch_size = batch_size or FUSION_BATCH
    n = planos.n_slices(vol.shape, plane)
    orig = planos.get_slice(vol, plane, 0).shape
    sq = planos.square_shape(orig, *planos.plane_spacing(plane, sp), sp[0])
    probs = np.zeros((3,) + tuple(vol.shape), dtype=np.uint8)
    fov = fov_center_mm(vol.shape, sp)                   # en val y en producción la línea media sale del campo de visión
    rng = np.random.default_rng(SEED)
    prep = {}

    def imagen(k):
        if k not in prep:
            prep[k] = _como_en_cache(planos._prepare_slice(vol, plane, k, sp, apply_hu_window, enhance_bone_contrast, letterbox_pair, IMAGE_SIZE))
        return prep[k]

    for s in range(0, n, batch_size):
        ids = list(range(s, min(s + batch_size, n)))
        xs = []
        for i in ids:
            chans = [imagen(k) for k in neighbor_indices(i, n, N_IMG)]
            if USE_POSITION_CHANNEL:
                chans.append(np.full((IMAGE_SIZE, IMAGE_SIZE), position_value(slice_position_mm(plane, i, sp), fov), dtype=np.float32))
            xs.append(np.stack(chans).astype(np.float32))
        p = _forward_lote(model, np.stack(xs), rng)
        for j, i in enumerate(ids):
            back = _restaurar_probs(p[j], sq, orig, IMAGE_SIZE)
            for c in range(3):
                planos.put_slice(probs[c], plane, i, back[c])
        for k in [k for k in prep if k < ids[0] - 1]:
            del prep[k]
    return probs


def _fila_caso(case_id):
    return df[df["case_id"] == case_id].iloc[0]


def probs_de_caso(case_id, vol, sp, planes=PLANES, modelos=None):
    """Probabilidades por plano de un caso: las lee del disco si ya están (mismo modelo) y si no las infiere y las guarda. -> {plano: memmap}"""
    modelos = modelos if modelos is not None else {}
    out = {}
    for plane in planes:
        f = PROB_DIR / f"{case_id}_{plane}_{_sha1(ruta_modelo(plane))}.npy"
        if not f.exists():
            configurar_plano(plane, False)
            if plane not in modelos:
                modelos[plane], _ = load_trained_model(ruta_modelo(plane))
            t = time.perf_counter()
            pr = inferir_volumen(modelos[plane], plane, vol, sp)
            np.save(f, pr)
            log(f"[fusión] caso {case_id} | {plane}: {planos.n_slices(vol.shape, plane)} cortes inferidos en {time.perf_counter() - t:.0f} s ({pr.nbytes / 1e6:.0f} MB)")
            del pr
        out[plane] = np.load(f, mmap_mode="r")
    return out


# ------------------------------------------------------------------------------------------ fusión
def _fusionar_prob(arrays, pesos):
    """arrays: {plano: [3, ...] uint8} con la misma forma -> etiquetas uint8 (0 fondo, 1-3 clases): argmax del promedio ponderado."""
    planes = [p for p, w in pesos.items() if w > 0]
    tot = float(sum(pesos[p] for p in planes))
    acc = None
    for p in planes:
        a = np.asarray(arrays[p], dtype=np.float32) * (pesos[p] / tot)
        acc = a if acc is None else acc + a
    stack = np.concatenate([(255.0 - acc.sum(0))[None], acc], axis=0)       # el fondo es 255 - suma de las 3 clases
    return stack.argmax(0).astype(np.uint8)


def etiquetas_fusion(probs, pesos, chunk=24):
    planes = [p for p, w in pesos.items() if w > 0]
    shape = probs[planes[0]].shape[1:]
    out = np.zeros(shape, dtype=np.uint8)
    for z0 in range(0, shape[0], chunk):
        z1 = min(shape[0], z0 + chunk)
        out[z0:z1] = _fusionar_prob({p: probs[p][:, z0:z1] for p in planes}, pesos)
    return out


# ------------------------------------------------------------------------------------------ métricas 3D
def _bbox(mask, margen=2):
    idx = [np.flatnonzero(mask.any(axis=tuple(a for a in range(3) if a != ax))) for ax in range(3)]
    if any(len(i) == 0 for i in idx):
        return None
    return tuple(slice(max(0, int(i[0]) - margen), int(i[-1]) + 1 + margen) for i in idx)


def _conteos_dice(gsem, lab):
    return {c: (float(((lab == c) & (gsem == c)).sum()), float((lab == c).sum()), float((gsem == c).sum())) for c in (1, 2, 3)}


def _dice(i, p, g):
    return 2 * i / (p + g) if (p + g) > 0 else float("nan")


def componentes_3d(lab):
    """Componentes conexos (26 vecinos) de cada clase -> {clase: (etiquetas int32, tamaños en vóxeles)}."""
    out = {}
    for c in (1, 2, 3):
        m = lab == c
        if m.any():
            cc, _ = ndi.label(m, structure=_STRUCT26)
            out[c] = (cc, np.bincount(cc.ravel()))
    return out


def instancias_desde_cc(ccs, shape, min_vox):
    """Quita los componentes de menos de min_vox vóxeles y numera el resto. -> (instancias int32, clase de cada id)."""
    inst = np.zeros(shape, dtype=np.int32)
    cls, n = [0], 0
    for c, (cc, sizes) in ccs.items():
        keep = np.flatnonzero(sizes >= max(1, min_vox))
        keep = keep[keep > 0]
        if len(keep) == 0:
            continue
        remap = np.zeros(len(sizes), dtype=np.int32)
        remap[keep] = np.arange(n + 1, n + 1 + len(keep), dtype=np.int32)
        inst = np.where(remap[cc] > 0, remap[cc], inst)
        cls += [c] * len(keep)
        n += len(keep)
    return inst, np.asarray(cls)


def _clase_de_id(i):                                       # 1-10 sacro, 11-20 coxal izq., 21-30 coxal der.
    return 1 if i <= 10 else 2 if i <= 20 else 3


def emparejar_3d(gt_inst, pred_inst, pred_cls, iou_min=0.5):
    """Fragmentos 3D: emparejamiento uno a uno (húngaro), misma región, IoU >= iou_min.
    -> dict con tp, suma de IoU de los aciertos, n_gt, n_pred y conteos por región."""
    m = (gt_inst > 0) | (pred_inst > 0)
    g, p = gt_inst[m].astype(np.int64), pred_inst[m].astype(np.int64)
    g_ids, p_ids = np.unique(g[g > 0]), np.unique(p[p > 0])
    res = {"tp": 0, "iou_sum": 0.0, "n_gt": int(len(g_ids)), "n_pred": int(len(p_ids))}
    for c in (1, 2, 3):
        res[f"n_gt_c{c}"] = int(sum(_clase_de_id(int(i)) == c for i in g_ids))
        res[f"n_pred_c{c}"] = int(sum(int(pred_cls[int(i)]) == c for i in p_ids))
    if len(g_ids) == 0 or len(p_ids) == 0:
        return res
    gs, ps = np.bincount(g), np.bincount(p, minlength=int(p_ids.max()) + 1)
    P = int(p.max()) + 1
    uk, cnt = np.unique(g * P + p, return_counts=True)
    gi = {int(v): k for k, v in enumerate(g_ids)}
    pi = {int(v): k for k, v in enumerate(p_ids)}
    iou = np.zeros((len(g_ids), len(p_ids)))
    for key, inter in zip(uk, cnt):
        a, b = int(key // P), int(key % P)
        if a == 0 or b == 0 or _clase_de_id(a) != int(pred_cls[b]):
            continue
        iou[gi[a], pi[b]] = inter / (gs[a] + ps[b] - inter)
    r, c = linear_sum_assignment(-iou)
    ok = [float(iou[a, b]) for a, b in zip(r, c) if iou[a, b] >= iou_min]
    res["tp"], res["iou_sum"] = len(ok), float(sum(ok))
    return res


def _resumen_frag(f):
    tp, g, p = int(f["tp"].sum()), int(f["n_gt"].sum()), int(f["n_pred"].sum())
    fp, fn = p - tp, g - tp
    sq = f["iou_sum"].sum() / tp if tp else float("nan")
    rq = tp / (tp + 0.5 * fp + 0.5 * fn) if (tp + fp + fn) else float("nan")
    d = f["n_pred"] - f["n_gt"]
    return {"fragmentos_reales": g, "fragmentos_predichos": p, "predichos/reales": p / g if g else float("nan"),
            "precision": tp / p if p else float("nan"), "recall": tp / g if g else float("nan"),
            "SQ": sq, "RQ(F1)": rq, "PQ": sq * rq, "sesgo por paciente (+ = de más)": float(d.mean()), "error abs. por paciente": float(d.abs().mean())}


# ------------------------------------------------------------------------------------------ corrida completa
def fusion_3d(split="val", variantes=None, max_casos=None, confirmar_test=False, planes=PLANES):
    """Infiere (o lee del disco) los 3 planos de cada paciente del split, fusiona y mide en 3D. Devuelve un dict con las tablas."""
    if split == "test" and not confirmar_test:
        raise RuntimeError("El split 'test' se evalúa UNA sola vez, al final. Llamar con confirmar_test=True cuando todo esté congelado.")
    variantes = variantes or VARIANTES
    casos = df[df["split"] == split]["case_id"].tolist()[:max_casos]
    log(f"[fusión] split {split}: {len(casos)} pacientes | variantes: {list(variantes)}")
    modelos = {}
    por_caso, frag_rows, vac = [], [], {v: [0, 0] for v in variantes}
    glob = {v: {c: [0.0, 0.0, 0.0] for c in (1, 2, 3)} for v in variantes}
    glob_limpio = {v: {c: [0.0, 0.0, 0.0] for c in (1, 2, 3)} for v in variantes}
    t0 = time.perf_counter()
    for n_caso, cid in enumerate(casos, 1):
        row = _fila_caso(cid)
        vol, sp, _ = load_canonical(row["image_path"], is_label=False, resolver=resolve_repo_path, dtype="native")
        L, _, _ = load_canonical(row["label_path"], is_label=True, resolver=resolve_repo_path, dtype="native")
        probs = probs_de_caso(cid, vol, sp, planes, modelos)
        del vol
        gsem = planos.region_of_ids(L)
        vacios = ~(gsem > 0).any(axis=(1, 2))                                  # cortes axiales sin pelvis
        vox_mm3 = float(sp[0] * sp[1] * sp[2])
        for v, pesos in variantes.items():
            if any(p not in probs for p, w in pesos.items() if w > 0):
                continue
            lab = etiquetas_fusion(probs, pesos)
            cd = _conteos_dice(gsem, lab)
            fila = {"caso": cid, "variante": v}
            for c in (1, 2, 3):
                fila[f"Dice {CLASES[c]}"] = _dice(*cd[c])
                for k in range(3):
                    glob[v][c][k] += cd[c][k]
            vac[v][0] += int((vacios & (lab > 0).any(axis=(1, 2))).sum()); vac[v][1] += int(vacios.sum())
            if v in VARIANTES_FRAG:
                bb = _bbox((lab > 0) | (gsem > 0))
                if bb is not None:
                    lab_c, gi_c, gs_c = lab[bb], L[bb], gsem[bb]
                    ccs = componentes_3d(lab_c)
                    for mm3 in MIN_MM3_GRID:
                        inst, cls = instancias_desde_cc(ccs, lab_c.shape, int(np.ceil(mm3 / vox_mm3)) if mm3 else 1)
                        r = emparejar_3d(gi_c, inst, cls)
                        frag_rows.append({"caso": cid, "variante": v, "min_mm3": mm3, **r})
                        if mm3 == MIN_MM3_LIMPIEZA:
                            lim = cls[inst]                                       # etiqueta de clase tras quitar lo pequeño
                            cl = _conteos_dice(gs_c, lim.astype(np.uint8))
                            for c in (1, 2, 3):
                                fila[f"Dice limpio {CLASES[c]}"] = _dice(*cl[c])
                                for k in range(3):
                                    glob_limpio[v][c][k] += cl[c][k]
                    del ccs
            por_caso.append(fila)
            del lab
            gc.collect()
        log(f"[fusión] {n_caso}/{len(casos)} pacientes | {time.perf_counter() - t0:.0f} s")
        del probs, L, gsem

    pc = pd.DataFrame(por_caso)
    dcols = [f"Dice {CLASES[c]}" for c in (1, 2, 3)]
    pc["Dice medio"] = pc[dcols].mean(axis=1)
    tabla = pd.DataFrame({v: {**{f"Dice {CLASES[c]}": _dice(*glob[v][c]) for c in (1, 2, 3)}} for v in variantes if v in set(pc["variante"])}).T
    tabla["Dice medio (agrupado)"] = tabla[dcols].mean(axis=1)
    tabla["Dice medio por paciente"] = pc.groupby("variante")["Dice medio"].mean()
    tabla["cortes axiales vacíos con predicción (%)"] = [100.0 * vac[v][0] / max(vac[v][1], 1) for v in tabla.index]
    if any(glob_limpio[v][1][2] > 0 for v in glob_limpio):
        for c in (1, 2, 3):
            tabla[f"Dice limpio {CLASES[c]}"] = [_dice(*glob_limpio[v][c]) if v in VARIANTES_FRAG else float("nan") for v in tabla.index]
    fr = pd.DataFrame(frag_rows)
    tabla_frag = (pd.DataFrame({(v, m): _resumen_frag(g) for (v, m), g in fr.groupby(["variante", "min_mm3"])}).T
                  if len(fr) else pd.DataFrame())
    pivot = pc.pivot(index="caso", columns="variante", values="Dice medio")[[v for v in variantes if v in set(pc["variante"])]]
    res = {"dice": tabla, "dice_por_paciente": pivot, "fragmentos_3d": tabla_frag, "detalle": pc, "split": split}
    RES_FUSION[split] = res
    FUSION_DIR.mkdir(parents=True, exist_ok=True)
    tabla.to_csv(FUSION_DIR / f"dice_3d_{split}.csv"); pivot.to_csv(FUSION_DIR / f"dice_por_paciente_{split}.csv")
    if len(tabla_frag):
        tabla_frag.to_csv(FUSION_DIR / f"fragmentos_3d_{split}.csv")
    print(f"\n===== Dice 3D ({split}, {len(casos)} pacientes; todas las cortes del volumen, también las vacías) =====")
    display(tabla.round(3))
    print("\nDice medio por paciente y variante:")
    display(pivot.round(3))
    if len(tabla_frag):
        print("\nFragmentos 3D (componentes conexos por clase; acierto = IoU >= 0.5 con un fragmento real de la misma región):")
        display(tabla_frag.round(3))
    return res


# ------------------------------------------------------------------------------------------ vista rápida
def mostrar_fusion(case_id, z=None, y=None, split="val", variantes=("axial", "coronal", "sagital", "3 planos (iguales)")):
    """Un corte axial y uno coronal del paciente: imagen | real | cada variante. Usa las probabilidades ya guardadas (corre fusion_3d antes)."""
    row = _fila_caso(case_id)
    vol, sp, _ = load_canonical(row["image_path"], is_label=False, resolver=resolve_repo_path, dtype="native")
    L, _, _ = load_canonical(row["label_path"], is_label=True, resolver=resolve_repo_path, dtype="native")
    probs = {p: np.load(PROB_DIR / f"{case_id}_{p}_{_sha1(ruta_modelo(p))}.npy", mmap_mode="r") for p in PLANES}
    gsem = planos.region_of_ids(L)
    if z is None:                                           # el corte axial / coronal con más fragmentos reales distintos
        z = int(np.argmax([len(np.unique(L[i])) for i in range(L.shape[0])]))
    if y is None:
        y = int(np.argmax([len(np.unique(L[:, j])) for j in range(L.shape[1])]))
    colores = np.array([[0, 0, 0], [230, 140, 20], [40, 110, 230], [40, 190, 100]], dtype=np.float32) / 255.0
    fig, axes = plt.subplots(2, 2 + len(variantes), figsize=(3.2 * (2 + len(variantes)), 6.6))
    for r, (plane, k) in enumerate((("axial", z), ("coronal", y))):
        im = apply_hu_window(np.asarray(planos.get_slice(vol, plane, k), dtype=np.float32))
        im = np.asarray(slice_to_square_display(im, plane, sp))
        origin = "upper" if plane == "axial" else "lower"

        def pintar(ax, lab2d, titulo):
            lab2d = slice_to_square_display(lab2d, plane, sp, label=True)
            rgb = np.stack([im] * 3, -1) * 0.8
            m = lab2d > 0
            rgb[m] = 0.45 * rgb[m] + 0.55 * colores[lab2d[m]]
            ax.imshow(np.clip(rgb, 0, 1), origin=origin); ax.set_title(titulo, fontsize=9); ax.axis("off")

        axes[r, 0].imshow(im, cmap="gray", origin=origin); axes[r, 0].set_title(f"{plane} {k} | caso {case_id}", fontsize=9); axes[r, 0].axis("off")
        pintar(axes[r, 1], planos.get_slice(gsem, plane, k), "real")
        for j, v in enumerate(variantes):
            pw = VARIANTES[v]
            sl = {p: (probs[p][:, k] if plane == "axial" else probs[p][:, :, k, :]) for p in pw}
            pintar(axes[r, 2 + j], _fusionar_prob(sl, pw), v)
    plt.tight_layout()
    FIG = FUSION_DIR / f"fusion_{case_id}.png"
    plt.savefig(FIG, dpi=110)
    plt.show()


def slice_to_square_display(arr, plane, sp, label=False):
    """Corte a píxeles cuadrados para dibujarlo (coronal/sagital tienen sz distinto de sx)."""
    return planos.slice_to_square(np.asarray(arr), plane, sp, label)


# ------------------------------------------------------------------------------------------ uso
# 1) prueba corta (un paciente; tarda ~1-3 min la primera vez, luego lee los volúmenes de probabilidad del disco):
#        RES = fusion_3d("val", max_casos=1)
# 2) todos los pacientes de validación:
#        RES = fusion_3d("val")
# 3) mirar un caso (corte axial y coronal con más fragmentos):
#        mostrar_fusion("035")
# 4) SOLO al final, con modelos y parámetros congelados:
#        fusion_3d("test", confirmar_test=True)


In [17]:
# ---------------------------------------------------------------------------
# DIAGNÓSTICO 3D de la fusión: (1) qué pasa con cada fragmento real; (2) cómo es el sacro por tercios y su punta.
# Pegar en una celda NUEVA después de celda_fusion_3d.py. Usa las probabilidades ya guardadas en disco (no infiere nada; ~1 min).
#
#   DIAG = diagnostico_fragmentos()                       # los 5 pacientes de validación, fusión de 3 planos, piezas >= 100 mm³
#   sacro_por_tercios()                                   # Dice del sacro en su tercio inferior / medio / superior y cuánto se acorta la punta
#
# Categorías de cada fragmento real (el predicho "dominante" es el componente 3D que más lo cubre, de la misma región):
#   acertado                       IoU >= 0.5 con su componente dominante
#   FUNDIDO con otro fragmento     el componente dominante lo cubre (>= 50 %) pero también cubre >= 50 % de otro fragmento real
#                                  (o es >= 2 veces más grande que él): la segmentación por clase es buena, pero no hay hueco que los separe
#   PARTIDO en varios              lo cubren >= 2 componentes (>= 10 % cada uno) y ninguno llega a IoU 0.5
#   NO DETECTADO / incompleto      menos de la mitad de sus vóxeles se predijo como su región
#   otro (IoU bajo)                lo demás (p. ej. predicho con exceso)
# ---------------------------------------------------------------------------
from IPython.display import display


def _etiq_caso(case_id, variante):
    pw = VARIANTES[variante]
    probs = {p: np.load(PROB_DIR / f"{case_id}_{p}_{_sha1(ruta_modelo(p))}.npy", mmap_mode="r") for p, w in pw.items() if w > 0}
    return etiquetas_fusion(probs, pw)


def _categorizar(gt_inst, pred_inst, pred_cls, lab):
    """Una fila por fragmento real. gt_inst: ids reales 1-30; pred_inst/pred_cls: componentes 3D predichos; lab: etiqueta por clase (0-3)."""
    m = (gt_inst > 0) & (pred_inst > 0)
    g, p = gt_inst[m].astype(np.int64), pred_inst[m].astype(np.int64)
    P = int(max(pred_inst.max(), 0)) + 1
    ov = {}
    if len(g):
        uk, cnt = np.unique(g * P + p, return_counts=True)
        ov = {(int(k // P), int(k % P)): int(c) for k, c in zip(uk, cnt)}
    gs = np.bincount(gt_inst.ravel().astype(np.int64), minlength=31)
    ps = np.bincount(pred_inst.ravel().astype(np.int64), minlength=P)
    mm = gt_inst > 0
    por_clase = np.bincount(gt_inst[mm].astype(np.int64) * 4 + lab[mm], minlength=31 * 4)
    filas = []
    for gid in [i for i in range(1, 31) if gs[i] > 0]:
        cl = _clase_de_id(gid)
        cobertura = por_clase[gid * 4 + cl] / gs[gid]                       # fracción de sus vóxeles predicha como su región
        mios = {pp: c for (gg, pp), c in ov.items() if gg == gid and int(pred_cls[pp]) == cl}
        dom = max(mios, key=mios.get) if mios else None
        iou = mios[dom] / (gs[gid] + ps[dom] - mios[dom]) if dom is not None else 0.0
        otros = [h for h in range(1, 31) if h != gid and gs[h] > 0 and _clase_de_id(h) == cl and dom is not None
                 and ov.get((h, dom), 0) / gs[h] >= 0.5]
        partes = [pp for pp, c in mios.items() if c / gs[gid] >= 0.10]
        cubre = mios[dom] / gs[gid] if dom is not None else 0.0
        if iou >= 0.5:
            cat = "acertado"
        elif cobertura < 0.5:
            cat = "NO DETECTADO / incompleto"
        elif dom is not None and cubre >= 0.5 and (otros or ps[dom] >= 2 * gs[gid]):
            cat = "FUNDIDO con otro fragmento"
        elif len(partes) >= 2:
            cat = "PARTIDO en varios"
        else:
            cat = "otro (IoU bajo)"
        filas.append({"id": gid, "región": CLASES[cl], "cobertura de vóxeles": round(float(cobertura), 3), "IoU con su componente": round(float(iou), 3),
                      "componentes que lo tocan": len(partes), "categoría": cat, "fundido con (ids reales)": ",".join(map(str, otros)), "vox": int(gs[gid])})
    return filas


def diagnostico_fragmentos(variante="3 planos (iguales)", min_mm3=100, split="val", casos=None):
    casos = casos or df[df["split"] == split]["case_id"].tolist()
    filas = []
    for cid in casos:
        row = _fila_caso(cid)
        L, sp, _ = load_canonical(row["label_path"], is_label=True, resolver=resolve_repo_path, dtype="native")
        lab = _etiq_caso(cid, variante)
        bb = _bbox((lab > 0) | (L > 0))
        if bb is None:
            continue
        lab_c, L_c = lab[bb], L[bb]
        inst, cls = instancias_desde_cc(componentes_3d(lab_c), lab_c.shape, int(np.ceil(min_mm3 / float(sp[0] * sp[1] * sp[2]))) if min_mm3 else 1)
        vox_mm3 = float(sp[0] * sp[1] * sp[2])
        for f in _categorizar(L_c, inst, cls, lab_c):
            f["caso"] = cid
            f["volumen mm³"] = round(f.pop("vox") * vox_mm3)
            filas.append(f)
    d = pd.DataFrame(filas)[["caso", "id", "región", "volumen mm³", "cobertura de vóxeles", "IoU con su componente", "componentes que lo tocan",
                             "categoría", "fundido con (ids reales)"]]
    bins = pd.cut(d["volumen mm³"], [0, 500, 5000, np.inf], labels=["< 500 mm³", "500-5000 mm³", "> 5000 mm³"])
    print(f"===== Qué pasa con cada fragmento real ({variante}, piezas predichas >= {min_mm3} mm³; {len(d)} fragmentos reales) =====")
    display(d["categoría"].value_counts().to_frame("fragmentos"))
    print("\nSegún el tamaño del fragmento real:")
    display(pd.crosstab(bins, d["categoría"]))
    print("\nFragmentos NO acertados (de mayor a menor volumen):")
    display(d[d["categoría"] != "acertado"].sort_values("volumen mm³", ascending=False).reset_index(drop=True))
    FUSION_DIR.mkdir(parents=True, exist_ok=True)
    d.to_csv(FUSION_DIR / f"diagnostico_fragmentos_{split}.csv", index=False)
    return d


def sacro_por_tercios(variantes=("axial", "coronal", "sagital", "3 planos (iguales)"), split="val", casos=None, min_mm3=100):
    """Dice del sacro dentro de la extensión real del sacro (en z), partida en tercios inferior / medio / superior, y cuánto sube la punta
    inferior predicha respecto a la real (mm; positivo = la predicción empieza más arriba = punta recortada)."""
    casos = casos or df[df["split"] == split]["case_id"].tolist()
    nombres = ("inferior (punta)", "medio", "superior")
    acc = {v: {t: [0.0, 0.0, 0.0] for t in nombres} for v in variantes}
    punta = {v: {} for v in variantes}
    for cid in casos:
        row = _fila_caso(cid)
        L, sp, _ = load_canonical(row["label_path"], is_label=True, resolver=resolve_repo_path, dtype="native")
        gsem = planos.region_of_ids(L)
        zs = np.flatnonzero((gsem == 1).any(axis=(1, 2)))
        if len(zs) == 0:
            continue
        z0, z1 = int(zs[0]), int(zs[-1])                                      # z crece hacia superior: z0 es la punta
        ed = np.linspace(z0, z1 + 1, 4).astype(int)
        min_vox = int(np.ceil(min_mm3 / float(sp[0] * sp[1] * sp[2])))
        for v in variantes:
            lab = _etiq_caso(cid, v)
            for k, t in enumerate(nombres):
                sl = slice(ed[k], ed[k + 1])
                p_, g_ = lab[sl] == 1, gsem[sl] == 1
                acc[v][t][0] += float((p_ & g_).sum()); acc[v][t][1] += float(p_.sum()); acc[v][t][2] += float(g_.sum())
            cc, _ = ndi.label(lab == 1, structure=_STRUCT26)
            sizes = np.bincount(cc.ravel())
            ok = np.flatnonzero(sizes >= max(1, min_vox)); ok = ok[ok > 0]
            keep = np.isin(cc, ok) if len(ok) else np.zeros(lab.shape, bool)
            zp = np.flatnonzero(keep.any(axis=(1, 2)))
            punta[v][cid] = float((int(zp[0]) - z0) * sp[2]) if len(zp) else float("nan")
    t1 = pd.DataFrame({v: {t: _dice(*acc[v][t]) for t in nombres} for v in variantes}).T
    t2 = pd.DataFrame(punta)
    t2.loc["media"] = t2.mean()
    print("===== Dice del sacro por tercios de su extensión real (sacro agrupado de todos los pacientes) =====")
    display(t1.round(3))
    print("\nPunta inferior: cuántos mm más ARRIBA empieza el sacro predicho que el real, por paciente (positivo = punta recortada):")
    display(t2.T.round(1))
    return t1, t2


# ------------------------------------------------------------------------------------------ uso


In [18]:
# ---------------------------------------------------------------------------
# DISTANCIA 3D al fragmento principal, medida con la función del equipo (src/distancias_fragmentos.measure_fragment_separations),
# sobre la reconstrucción fusionada de 3 planos, y comparada con la distancia de las etiquetas reales.
# Pegar en una celda NUEVA después de celda_fusion_3d.py y celda_diagnostico_3d.py (usa las probabilidades ya guardadas; ~1-2 min).
#
#   DIST = distancias_val()                       # val, 3 planos, piezas >= 100 mm³
#   DIST = distancias_val(min_mm3=500)            # otro tamaño mínimo
#
# Cómo se mide (igual que distancias_referencia.csv): distancia mínima borde a borde entre máscaras voxelizadas, en mm; 0 = se tocan
# (por cara, arista o esquina). El fragmento principal es el de más vóxeles de su región. En la predicción los fragmentos son los
# componentes conexos 3D de cada región (los mismos de fusion_3d), así que un fragmento solo aparece como pieza propia si la
# segmentación deja un hueco con el hueso vecino.
# ---------------------------------------------------------------------------
from src.distancias_fragmentos import measure_fragment_separations

_REGION_NOMBRE = {1: "sacro", 2: "coxal_izquierdo", 3: "coxal_derecho"}


def _pares_3d(gt_inst, pred_inst, pred_cls, iou_min=0.5):
    """Emparejamiento uno a uno (húngaro), misma región, IoU >= iou_min. -> {id real: (id predicho, IoU)}."""
    m = (gt_inst > 0) & (pred_inst > 0)
    g, p = gt_inst[m].astype(np.int64), pred_inst[m].astype(np.int64)
    if len(g) == 0:
        return {}
    g_ids, p_ids = np.unique(g), np.unique(p)
    gs = np.bincount(gt_inst.ravel().astype(np.int64))
    ps = np.bincount(pred_inst.ravel().astype(np.int64))
    P = int(p.max()) + 1
    uk, cnt = np.unique(g * P + p, return_counts=True)
    gi = {int(v): k for k, v in enumerate(g_ids)}
    pi = {int(v): k for k, v in enumerate(p_ids)}
    iou = np.zeros((len(g_ids), len(p_ids)))
    for key, inter in zip(uk, cnt):
        a, b = int(key // P), int(key % P)
        if _clase_de_id(a) != int(pred_cls[b]):
            continue
        iou[gi[a], pi[b]] = inter / (gs[a] + ps[b] - inter)
    r, c = linear_sum_assignment(-iou)
    return {int(g_ids[a]): (int(p_ids[b]), float(iou[a, b])) for a, b in zip(r, c) if iou[a, b] >= iou_min}


def distancias_val(variante="3 planos (iguales)", min_mm3=100, split="val", casos=None):
    casos = casos or df[df["split"] == split]["case_id"].tolist()
    filas, fps = [], []
    for cid in casos:
        row = _fila_caso(cid)
        L, sp, _ = load_canonical(row["label_path"], is_label=True, resolver=resolve_repo_path, dtype="native")
        lab = _etiq_caso(cid, variante)
        bb = _bbox((lab > 0) | (L > 0))
        if bb is None:
            continue
        lab_c, L_c = lab[bb], L[bb]
        vox_mm3 = float(sp[0] * sp[1] * sp[2])
        sp3 = tuple(float(s) for s in sp)                                           # (x, y, z) mm, como GetSpacing() de SimpleITK
        inst, cls = instancias_desde_cc(componentes_3d(lab_c), lab_c.shape, int(np.ceil(min_mm3 / vox_mm3)) if min_mm3 else 1)
        gt_tab = measure_fragment_separations(L_c.astype(np.int16), sp3, case_id=cid)
        if inst.max() > 0:
            reg = {i: _REGION_NOMBRE[int(cls[i])] for i in range(1, len(cls))}
            pr_tab = measure_fragment_separations(inst, sp3, case_id=cid, region_by_id=reg).set_index("fragment_id")
        else:
            pr_tab = pd.DataFrame(columns=["is_main", "distance_mm", "n_voxels"])
        pares = _pares_3d(L_c, inst, cls)
        for r in gt_tab.itertuples():
            par = pares.get(int(r.fragment_id))
            p_row = pr_tab.loc[par[0]] if par is not None else None
            filas.append({"caso": cid, "región": r.region, "id real": int(r.fragment_id), "principal real": bool(r.is_main),
                          "volumen mm³": round(r.n_voxels * vox_mm3), "distancia real (mm)": round(float(r.distance_mm), 2),
                          "detectado como pieza propia": par is not None, "IoU": round(par[1], 3) if par else np.nan,
                          "principal predicho": bool(p_row["is_main"]) if p_row is not None else np.nan,
                          "distancia predicha (mm)": round(float(p_row["distance_mm"]), 2) if p_row is not None else np.nan})
        usados = {v[0] for v in pares.values()}
        for pid, r in pr_tab.iterrows():
            if not r["is_main"] and int(pid) not in usados:
                fps.append({"caso": cid, "región": _REGION_NOMBRE[int(cls[int(pid)])], "volumen mm³": round(r["n_voxels"] * vox_mm3),
                            "distancia predicha (mm)": round(float(r["distance_mm"]), 2)})
    d = pd.DataFrame(filas)
    sec = d[~d["principal real"]].copy()
    sec["error (mm)"] = (sec["distancia predicha (mm)"] - sec["distancia real (mm)"]).where(sec["detectado como pieza propia"])
    sec["distancia real"] = pd.cut(sec["distancia real (mm)"], [-0.01, 0.0, 2.0, np.inf], labels=["0 mm (se toca)", "0-2 mm", "> 2 mm"])
    prin = d[d["principal real"]]
    print(f"===== Distancia al fragmento principal ({variante}, piezas >= {min_mm3} mm³, {split}: {len(casos)} pacientes) =====")
    print(f"Fragmentos principales: {len(prin)} | acertados como pieza propia: {int(prin['detectado como pieza propia'].sum())} "
          f"| y también principal en la predicción: {int((prin['principal predicho'] == True).sum())}")
    print(f"Fragmentos secundarios reales: {len(sec)}")
    tabla = sec.groupby("distancia real", observed=False).agg(**{"fragmentos": ("id real", "size"),
                                                                  "detectados": ("detectado como pieza propia", "sum")})
    tabla["% detectados"] = (100 * tabla["detectados"] / tabla["fragmentos"].where(tabla["fragmentos"] > 0)).round(0)
    print("\nSecundarios reales según su distancia real al principal, y cuántos salen como pieza propia en la predicción:")
    display(tabla)
    det = sec[sec["detectado como pieza propia"]]
    if len(det):
        print(f"\nSecundarios detectados: {len(det)} | error absoluto de la distancia: media {det['error (mm)'].abs().mean():.2f} mm, "
              f"mediana {det['error (mm)'].abs().median():.2f} mm, máximo {det['error (mm)'].abs().max():.2f} mm")
    else:
        print("\nNingún secundario detectado como pieza propia: no hay distancias predichas que comparar.")
    print("\nTodos los fragmentos secundarios reales:")
    display(sec.drop(columns=["distancia real", "principal real"]).sort_values(["caso", "id real"]).reset_index(drop=True))
    fp = pd.DataFrame(fps)
    print(f"\nPiezas predichas secundarias SIN pareja real (falsos positivos): {len(fp)}")
    if len(fp):
        display(fp)
    FUSION_DIR.mkdir(parents=True, exist_ok=True)
    d.to_csv(FUSION_DIR / f"distancias_val_{split}.csv", index=False)
    return d


# ------------------------------------------------------------------------------------------ uso


## 3. Usar los modelos de la etapa B

In [19]:
# ---------------------------------------------------------------------------
# Pasar la fusión a los modelos de la ETAPA B (best_B_decoder.pth). Correr DESPUÉS de las celdas de definiciones y ANTES de test.
# Exporta cada plano a un archivo ligero de inferencia (modelo_multitarea_{plano}_B.pt) y redirige ruta_modelo a esos archivos.
# ---------------------------------------------------------------------------
PLANOS_B = ("axial", "coronal", "sagital")


def ruta_modelo(plane):                                   # sobrescribe la de la celda de fusión: ahora apunta a los pesos B
    return MODELOS_DIR / f"entrenamiento_multitarea_{plane}" / f"modelo_multitarea_{plane}_B.pt"


for p in PLANOS_B:
    ck = MODELOS_DIR / f"entrenamiento_multitarea_{p}" / "best_B_decoder.pth"
    assert ck.exists(), f"falta {ck} (¿se entrenó el plano {p} en este equipo?)"
    dst = ruta_modelo(p)
    if not dst.exists():
        info = export_inference_checkpoint(ck, dst)["selected_from"]
    else:
        info = torch.load(dst, map_location="cpu", weights_only=True)["selected_from"]
    assert info["stage"] == "B_decoder", f"{p}: el archivo sale de la etapa {info['stage']}, no de B_decoder"
    load_trained_model(dst)                               # comprueba que se reconstruye bien
    print(f"{p:8s} -> {dst.name} | etapa {info['stage']} | época {info['epoch']}")


axial    -> modelo_multitarea_axial_B.pt | etapa B_decoder | época 10
coronal  -> modelo_multitarea_coronal_B.pt | etapa B_decoder | época 10
sagital  -> modelo_multitarea_sagital_B.pt | etapa B_decoder | época 9


## 4. TEST (correr una sola vez)

In [20]:
# ---------------------------------------------------------------------------
# TEST FINAL: se corre UNA sola vez, con todo congelado (modelos B, fusión de 3 planos con pesos iguales, fragmentos >= 500 mm³).
# No cambiar nada después de ver los resultados. Tarda ~1-2 min por paciente (la primera llamada infiere; las demás leen del disco).
# ---------------------------------------------------------------------------
MIN_MM3_FINAL = 500
RES_TEST = fusion_3d("test", confirmar_test=True)                          # Dice 3D, cortes vacíos y fragmentos, por variante
DIAG_TEST = diagnostico_fragmentos(split="test", min_mm3=MIN_MM3_FINAL)    # qué pasa con cada fragmento real
DIST_TEST = distancias_val(split="test", min_mm3=MIN_MM3_FINAL)            # distancia al fragmento principal vs la real
TERCIOS_TEST = sacro_por_tercios(split="test", min_mm3=MIN_MM3_FINAL)      # sacro por tercios y posición de la punta

import shutil
zipf = shutil.make_archive(str(FUSION_DIR.parent / "resultados_test_fusion"), "zip", root_dir=str(FUSION_DIR), base_dir=".")
print("\nResultados empaquetados (sin las probabilidades):", zipf)


[2026-10-17 19:36:54] [fusión] split test: 20 pacientes | variantes: ['axial', 'coronal', 'sagital', 'axial+coronal', 'axial+sagital', 'coronal+sagital', '3 planos (iguales)', '3 planos (axial x2)']


<string>:42: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
<string>:47: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)


[2026-10-17 19:37:06] [fusión] caso 003 | axial: 285 cortes inferidos en 9 s (167 MB)
[2026-10-17 19:37:16] [fusión] caso 003 | coronal: 381 cortes inferidos en 9 s (167 MB)
[2026-10-17 19:37:29] [fusión] caso 003 | sagital: 512 cortes inferidos en 13 s (167 MB)
[2026-10-17 19:37:59] [fusión] 1/20 pacientes | 65 s
[2026-10-17 19:38:12] [fusión] caso 005 | axial: 312 cortes inferidos en 9 s (245 MB)
[2026-10-17 19:38:22] [fusión] caso 005 | coronal: 512 cortes inferidos en 10 s (245 MB)
[2026-10-17 19:38:37] [fusión] caso 005 | sagital: 512 cortes inferidos en 15 s (245 MB)
[2026-10-17 19:39:12] [fusión] 2/20 pacientes | 138 s
[2026-10-17 19:39:25] [fusión] caso 010 | axial: 350 cortes inferidos en 10 s (275 MB)
[2026-10-17 19:39:36] [fusión] caso 010 | coronal: 512 cortes inferidos en 10 s (275 MB)
[2026-10-17 19:39:48] [fusión] caso 010 | sagital: 512 cortes inferidos en 12 s (275 MB)
[2026-10-17 19:40:30] [fusión] 3/20 pacientes | 216 s
[2026-10-17 19:40:42] [fusión] caso 012 | axial

,Dice sacro,Dice coxal izq.,Dice coxal der.,Dice medio (agrupado),Dice medio por paciente,cortes axiales vacíos con predicción (%),Dice limpio sacro,Dice limpio coxal izq.,Dice limpio coxal der.
axial,0.938,0.965,0.972,0.958,0.957,4.122,0.939,0.966,0.972
coronal,0.939,0.968,0.968,0.958,0.957,2.120,0.939,0.969,0.968
sagital,0.939,0.965,0.971,0.958,0.956,7.951,0.939,0.965,0.972
axial+coronal,0.953,0.973,0.976,0.968,0.966,1.060,NaN,NaN,NaN
axial+sagital,0.952,0.974,0.979,0.968,0.967,1.237,NaN,NaN,NaN
coronal+sagital,0.955,0.976,0.979,0.970,0.969,0.530,NaN,NaN,NaN
3 planos (iguales),0.959,0.978,0.981,0.973,0.972,0.824,0.959,0.978,0.981
3 planos (axial x2),0.956,0.976,0.979,0.970,0.969,1.413,NaN,NaN,NaN



Dice medio por paciente y variante:


...

===== Dice del sacro por tercios de su extensión real (sacro agrupado de todos los pacientes) =====


,inferior (punta),medio,superior
axial,0.904,0.934,0.947
coronal,0.888,0.941,0.942
sagital,0.902,0.933,0.947
3 planos (iguales),0.927,0.956,0.965



Punta inferior: cuántos mm más ARRIBA empieza el sacro predicho que el real, por paciente (positivo = punta recortada):


,003,005,010,012,020,028,029,031,033,038,...,044,045,057,061,075,078,081,089,097,media
axial,2.0,-5.0,0.0,0.0,-1.6,1.0,-1.6,0.0,-12.8,-2.4,...,-2.5,0.0,-5.0,0.8,6.0,-0.8,0.8,-1.0,0.0,-1.2
coronal,5.0,-1.0,3.2,19.0,-0.8,0.0,0.8,4.0,1.6,-3.2,...,-0.6,-1.0,-3.0,1.6,5.0,0.0,4.0,3.0,-1.0,1.7
sagital,2.0,-5.0,-0.8,6.0,0.0,-2.0,0.0,1.0,-12.8,-3.2,...,-0.6,-1.0,-3.0,0.8,7.0,0.8,0.8,0.0,10.0,-0.0
3 planos (iguales),3.0,-5.0,0.8,5.0,-0.8,1.0,0.8,0.0,-5.6,-3.2,...,-1.2,-1.0,-3.0,0.8,6.0,0.0,0.8,0.0,0.0,-0.2



Resultados empaquetados (sin las probabilidades): D:\proyecto_analitica_corte2\resultados\resultados_test_fusion.zip
